# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 1/12** (CPU, internet off). **1778 series**, ~10.7 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part1.zip`
- `manifest_rgb25d_320_part1.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 1
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260","series":"1.2.826.0.1.3680043.8.498.40734206102458723096154687147390476697","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231","series":"1.2.826.0.1.3680043.8.498.93288538765996574291399549037773479273","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339","series":"1.2.826.0.1.3680043.8.498.23891625731161491661628238457017424651","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606","series":"1.2.826.0.1.3680043.8.498.11928835253023108906781394031354721015","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394","series":"1.2.826.0.1.3680043.8.498.12085504130809929729103897247086930468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.87979090750692080284624781117007957118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866","series":"1.2.826.0.1.3680043.8.498.12421004035680984708227339687183062277","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401","series":"1.2.826.0.1.3680043.8.498.52617317497828611151327175969188939183","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736","series":"1.2.826.0.1.3680043.8.498.75225539867204936817329444857995249574","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.18034687025255390862101903338402067103","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365","series":"1.2.826.0.1.3680043.8.498.11826433858276834356507870940860092091","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097","series":"1.2.826.0.1.3680043.8.498.12352377770249024977650111343773350963","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","series":"1.2.826.0.1.3680043.8.498.66213850715195648479247228176806395878","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364","series":"1.2.826.0.1.3680043.8.498.61868030571167897231542039755015811539","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100","series":"1.2.826.0.1.3680043.8.498.25823386585191095678975577314110303662","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339","series":"1.2.826.0.1.3680043.8.498.96185036981695528593702399431876772612","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834","series":"1.2.826.0.1.3680043.8.498.11809311849299898951648714317269528811","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","series":"1.2.826.0.1.3680043.8.498.61113792412753681286453271342770134320","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471","series":"1.2.826.0.1.3680043.8.498.81744319704989210022852043593436430024","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314","series":"1.2.826.0.1.3680043.8.498.10627582734210013379904392398124774430","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586","series":"1.2.826.0.1.3680043.8.498.64427747362483464930949414855098205329","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019","series":"1.2.826.0.1.3680043.8.498.68536734261021460331290518059387895086","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205","series":"1.2.826.0.1.3680043.8.498.10180173995522375163398189474717875046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930","series":"1.2.826.0.1.3680043.8.498.70348539454982244256928330288311963066","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161","series":"1.2.826.0.1.3680043.8.498.12461975233583204950838091362146598280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10165669527316708372190215034240922433","series":"1.2.826.0.1.3680043.8.498.75993451826663109450667479575056158631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","series":"1.2.826.0.1.3680043.8.498.72842044325314136815194252497811679183","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.98532718918289001119806201719594950072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507","series":"1.2.826.0.1.3680043.8.498.91881351992022083557172940752490598777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773","series":"1.2.826.0.1.3680043.8.498.97125702020356409016261693917242009340","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775","series":"1.2.826.0.1.3680043.8.498.36856803420888463595920193264903955784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585","series":"1.2.826.0.1.3680043.8.498.73716062789315276725794612269915852462","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062","series":"1.2.826.0.1.3680043.8.498.11030074485821475144486026602788780384","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.12793186650850639025527456007385566346","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10240166440044087554182120832488098189","series":"1.2.826.0.1.3680043.8.498.11185277091793079476657652906176227029","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.11522915971551197221627494920191937549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531","series":"1.2.826.0.1.3680043.8.498.12578412793111528360044815662652935286","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286","series":"1.2.826.0.1.3680043.8.498.12427381584228298885293688963574323648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903","series":"1.2.826.0.1.3680043.8.498.24268032675274956886789362244354592047","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.33328952706795214580541249148048792819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029","series":"1.2.826.0.1.3680043.8.498.13073929499931293515978165461325174342","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639","series":"1.2.826.0.1.3680043.8.498.28709691394821527185562428188835532840","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669","series":"1.2.826.0.1.3680043.8.498.48521357060600788194545215388379604259","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.89596859812892749464024763130346035215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052","series":"1.2.826.0.1.3680043.8.498.28095032587022735220409990288365943063","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272","series":"1.2.826.0.1.3680043.8.498.13311523076500809713191034725045232703","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846","series":"1.2.826.0.1.3680043.8.498.11035294124377312501783856172465496025","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.12642773720081203807866549236267377452","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10322853084122920408377349136830993432","series":"1.2.826.0.1.3680043.8.498.82471067941955627352762897096494474163","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687","series":"1.2.826.0.1.3680043.8.498.68873891053134610128375538021215302734","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10336300544286006569730381727202519821","series":"1.2.826.0.1.3680043.8.498.11844188407552711789690283529214215166","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.12965087742880830068568521510508595247","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385","series":"1.2.826.0.1.3680043.8.498.24200387830413636724043981922736727063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197","series":"1.2.826.0.1.3680043.8.498.13267298163331145210366163371239959463","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10370612191212456931506340727148605247","series":"1.2.826.0.1.3680043.8.498.75322298236812688477618940983414341707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.24139970067000919877444819504826219769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500","series":"1.2.826.0.1.3680043.8.498.58196915908131229791409634721997202543","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341","series":"1.2.826.0.1.3680043.8.498.24366922563288150531040273712070930123","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933","series":"1.2.826.0.1.3680043.8.498.25302692399332257603187161185607860127","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007","series":"1.2.826.0.1.3680043.8.498.50634149790334164775892444884588459818","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","series":"1.2.826.0.1.3680043.8.498.59464412948897152519910809718807224130","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667","series":"1.2.826.0.1.3680043.8.498.94214516312503971502643459522199141119","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029","series":"1.2.826.0.1.3680043.8.498.95936759811798945730462416281916454951","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.75290556037896370144248993657476895450","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399","series":"1.2.826.0.1.3680043.8.498.29579190378448332893530282756381391902","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069","series":"1.2.826.0.1.3680043.8.498.48078024353132221856402718520085872787","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.77044855060417156547958783987043808193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397","series":"1.2.826.0.1.3680043.8.498.12557100315402165352566128033527350084","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674","series":"1.2.826.0.1.3680043.8.498.36793679469803569057579621244248368744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884","series":"1.2.826.0.1.3680043.8.498.10055871012601176668266024881216265472","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546","series":"1.2.826.0.1.3680043.8.498.66123899265825770149785331924943179670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584","series":"1.2.826.0.1.3680043.8.498.25396033023077959312411814655577640918","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716","series":"1.2.826.0.1.3680043.8.498.11400113938078360738668746941185178006","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078","series":"1.2.826.0.1.3680043.8.498.92854885655383840904925782352020836991","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522","series":"1.2.826.0.1.3680043.8.498.13273707243929656401793364347035371808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10521372086358356219840179458708706015","series":"1.2.826.0.1.3680043.8.498.42006282073220349563312207314700700586","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196","series":"1.2.826.0.1.3680043.8.498.79037672002431161944195429269545776239","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739","series":"1.2.826.0.1.3680043.8.498.18432266042235455549443291880412428631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953","series":"1.2.826.0.1.3680043.8.498.93743879713071798854638156048315549541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573","series":"1.2.826.0.1.3680043.8.498.11660334315706083522118701228233492061","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10541029960718591409318399557079055927","series":"1.2.826.0.1.3680043.8.498.16915540365376072609391666864990589257","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216","series":"1.2.826.0.1.3680043.8.498.12537729854295691260415214247516064738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10549360192675575195491716398555148512","series":"1.2.826.0.1.3680043.8.498.50285978821899612901393331984140222031","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109","series":"1.2.826.0.1.3680043.8.498.18192030546439927111861649455897467268","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10580260109117947709069163110100632085","series":"1.2.826.0.1.3680043.8.498.13040886496802701812926538181866303692","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145","series":"1.2.826.0.1.3680043.8.498.68011373527768550967840462443388575061","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893","series":"1.2.826.0.1.3680043.8.498.77636139070295531239702483298199972343","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137","series":"1.2.826.0.1.3680043.8.498.10171556843385534660043075095232811854","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146","series":"1.2.826.0.1.3680043.8.498.71230382837592373156772659535538481532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","series":"1.2.826.0.1.3680043.8.498.27144880443969474425725114698732083939","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403","series":"1.2.826.0.1.3680043.8.498.10730915732703090825891907913890601582","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430","series":"1.2.826.0.1.3680043.8.498.36573843666696173049026764641712105065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.37550004584356291649744033589967316474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477","series":"1.2.826.0.1.3680043.8.498.20963753454661001170248974893293100175","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392","series":"1.2.826.0.1.3680043.8.498.29425682987584059092317540568150723095","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374","series":"1.2.826.0.1.3680043.8.498.54773029728329248565717427491594479947","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","series":"1.2.826.0.1.3680043.8.498.20040198002208537788040456403423745793","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.10943910845018484217338945109491283099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082","series":"1.2.826.0.1.3680043.8.498.10538999233263503648504657145384035327","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731","series":"1.2.826.0.1.3680043.8.498.95028627272350851415654324944075548997","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","series":"1.2.826.0.1.3680043.8.498.78230812575687674002335122448562904994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071","series":"1.2.826.0.1.3680043.8.498.20363333326596567038579935247878308902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10699348333280906956391277892869999716","series":"1.2.826.0.1.3680043.8.498.12974120134089351731803563856928178455","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.37874725816900486489273730146280036454","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879","series":"1.2.826.0.1.3680043.8.498.46634512196946932804071739100758949107","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895","series":"1.2.826.0.1.3680043.8.498.10534246449281518556722898665605526094","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732","series":"1.2.826.0.1.3680043.8.498.12617821363683523179208278133648742798","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.33017584061737095643899116479395964796","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10750145216793948201921667945805555093","series":"1.2.826.0.1.3680043.8.498.55305594432109919924095982887746552936","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628","series":"1.2.826.0.1.3680043.8.498.98070138113625740519169011295751319453","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.10116662845773324052026000780995686078","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.51228188889386157203931223753014577896","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","series":"1.2.826.0.1.3680043.8.498.13281845911861668810236972504418892573","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10774147135673365265606645781662088745","series":"1.2.826.0.1.3680043.8.498.12482625042937463915258655493577144001","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770","series":"1.2.826.0.1.3680043.8.498.27882670850180881984580027854091462362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450","series":"1.2.826.0.1.3680043.8.498.10902309271785780903009242072477327622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406","series":"1.2.826.0.1.3680043.8.498.69848452678164681779808628771850858130","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597","series":"1.2.826.0.1.3680043.8.498.22572236264755677603442961370513577655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306","series":"1.2.826.0.1.3680043.8.498.62929080054207437683301009401123490901","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","series":"1.2.826.0.1.3680043.8.498.77748016361476875996647003573195171078","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.32272734202654093192896786802731816876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318","series":"1.2.826.0.1.3680043.8.498.32371525867419976866348437967698442743","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766","series":"1.2.826.0.1.3680043.8.498.68281426200463062195912083374274430088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","series":"1.2.826.0.1.3680043.8.498.70276596036299708991890505073868853321","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385","series":"1.2.826.0.1.3680043.8.498.48258812735362359930403305604321211278","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021","series":"1.2.826.0.1.3680043.8.498.16915958754993425297867872920350309193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10890699078723972693302029683905114834","series":"1.2.826.0.1.3680043.8.498.91347003748391899540001900316150369483","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.12797443673236930975220436155240348846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466","series":"1.2.826.0.1.3680043.8.498.23504633479098554642941085243835025076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235","series":"1.2.826.0.1.3680043.8.498.29562603463668503307708628026529243535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902","series":"1.2.826.0.1.3680043.8.498.11055208927178096623715671958092370751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403","series":"1.2.826.0.1.3680043.8.498.21019537848955756955587117165880537638","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737","series":"1.2.826.0.1.3680043.8.498.14199549614186636963555079754175501364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","series":"1.2.826.0.1.3680043.8.498.11871893868980567714339311441974902370","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.85953770508206709993263761703626383746","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514","series":"1.2.826.0.1.3680043.8.498.74778607439151213114944611985465291865","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245","series":"1.2.826.0.1.3680043.8.498.89173702781233879031927550081442912069","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528","series":"1.2.826.0.1.3680043.8.498.12866977000580477967061693818187797686","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","series":"1.2.826.0.1.3680043.8.498.11515525514613269417702093064558155658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100","series":"1.2.826.0.1.3680043.8.498.29414572606361747732764096297287823836","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.28941279603892704447348805971872761009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054","series":"1.2.826.0.1.3680043.8.498.33270308871959676062038358998103393620","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510","series":"1.2.826.0.1.3680043.8.498.90615948084786827528167792650334160680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286","series":"1.2.826.0.1.3680043.8.498.59061070729356625756784116221406815925","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","series":"1.2.826.0.1.3680043.8.498.12766104373421817848078121167935599611","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043543420859531726426521315399258528","series":"1.2.826.0.1.3680043.8.498.46147724362450143137389212614658010316","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545","series":"1.2.826.0.1.3680043.8.498.25242199583610751323398221132559743173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860","series":"1.2.826.0.1.3680043.8.498.12082796582322702335917928294626349997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026","series":"1.2.826.0.1.3680043.8.498.42048326390002349917456936471800419634","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089","series":"1.2.826.0.1.3680043.8.498.30208525097290665927152085422006098205","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995","series":"1.2.826.0.1.3680043.8.498.10825384059994367081012332056122782237","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635","series":"1.2.826.0.1.3680043.8.498.12323229318069509142385539896983676281","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233","series":"1.2.826.0.1.3680043.8.498.12861573446379572716532501940517027434","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427","series":"1.2.826.0.1.3680043.8.498.84521675592173390620495224701596824603","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324","series":"1.2.826.0.1.3680043.8.498.12141436334054713914232539611515395634","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.10913512382904416605362249525983160244","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11128099896284546522122813576483001027","series":"1.2.826.0.1.3680043.8.498.50371199093890552104945399877481999296","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219","series":"1.2.826.0.1.3680043.8.498.12830423400827094776800332052866971861","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286","series":"1.2.826.0.1.3680043.8.498.70301303950014765735858554361488355539","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960","series":"1.2.826.0.1.3680043.8.498.85715432931154305628522012939812201324","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456","series":"1.2.826.0.1.3680043.8.498.83145314821591696340491842120234568175","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.27354094571597281567661550362861625952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656","series":"1.2.826.0.1.3680043.8.498.11761715950370563122457836738371378772","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997","series":"1.2.826.0.1.3680043.8.498.96596248587176077048646375432950457868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863","series":"1.2.826.0.1.3680043.8.498.75353553084433738305211497573128050980","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","series":"1.2.826.0.1.3680043.8.498.75821181738985483720264099454678078601","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.12667198095796557437928234362388060048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208663314213048426907645330143931419","series":"1.2.826.0.1.3680043.8.498.75532322911856426490566819507498965987","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11211598263863419448236436640277320409","series":"1.2.826.0.1.3680043.8.498.24764255313329471182002636264973414185","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","series":"1.2.826.0.1.3680043.8.498.99921849181460843543024533935216237206","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115","series":"1.2.826.0.1.3680043.8.498.84114162145930932790168836750135767077","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867","series":"1.2.826.0.1.3680043.8.498.98232507967890312798436479076030524475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11231822445970527276203191047278744341","series":"1.2.826.0.1.3680043.8.498.46212942445284851072062385113352241319","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007","series":"1.2.826.0.1.3680043.8.498.12378914010937184776968522123902927764","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974","series":"1.2.826.0.1.3680043.8.498.42720448797758257387608414933291617830","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11262944211104069231500934582412003273","series":"1.2.826.0.1.3680043.8.498.53365159762076500398919470493843628202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","series":"1.2.826.0.1.3680043.8.498.12598169077787511889922564887005305814","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.52634596682552762026530520017838606747","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353","series":"1.2.826.0.1.3680043.8.498.37083395733336631223619282238408836800","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529","series":"1.2.826.0.1.3680043.8.498.47682560429774372357123325004419780690","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580","series":"1.2.826.0.1.3680043.8.498.14912947740172635156351119156260949957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.11787809568620979491270258966474553519","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393","series":"1.2.826.0.1.3680043.8.498.32914721834344564941589017719515617486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","series":"1.2.826.0.1.3680043.8.498.13345390228881908219640159343417880503","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.29393702732276002945262706598485766066","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712","series":"1.2.826.0.1.3680043.8.498.11359013489573686068200970512813186439","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072","series":"1.2.826.0.1.3680043.8.498.12616636289723947946455686235985091594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260","series":"1.2.826.0.1.3680043.8.498.56228555291829565204943336150894348098","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11352722814008170661344291337061231555","series":"1.2.826.0.1.3680043.8.498.99452992207247220372945797505612214412","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241","series":"1.2.826.0.1.3680043.8.498.13397418997381548880920459500334445458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.10450090035285223893337909203260865351","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.40578720090280485592794345356779543757","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749","series":"1.2.826.0.1.3680043.8.498.12329970771927978973884161838434558922","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602","series":"1.2.826.0.1.3680043.8.498.17121135671640760608409823070709776798","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512","series":"1.2.826.0.1.3680043.8.498.49436757509612000304748940967385111775","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11420670844338686032030860787688858057","series":"1.2.826.0.1.3680043.8.498.91874901198946499590296363841626051878","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.98035715761706714542695628963338758036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794","series":"1.2.826.0.1.3680043.8.498.40016912542098410354330232389946664224","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797","series":"1.2.826.0.1.3680043.8.498.10864644352720686730051739366125697625","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234","series":"1.2.826.0.1.3680043.8.498.18902829426376861333471178576807692551","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193","series":"1.2.826.0.1.3680043.8.498.10904216568631663771256931394597404918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.18928762903171529819354140506084437519","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592","series":"1.2.826.0.1.3680043.8.498.30442580964171585737410182698737976948","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","series":"1.2.826.0.1.3680043.8.498.68604870128733316834640806017973955004","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11477583622065195478886525904745372063","series":"1.2.826.0.1.3680043.8.498.53722508361270299870776677858085921303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099","series":"1.2.826.0.1.3680043.8.498.12381096939840551428260590839402696166","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.56833988135807589908860014554723339127","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815","series":"1.2.826.0.1.3680043.8.498.10322983986612959352213669104230746047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726","series":"1.2.826.0.1.3680043.8.498.10272233408824746822209114908680885633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11506856887791286395737465774061395929","series":"1.2.826.0.1.3680043.8.498.99916252161527842562250604470337191039","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664","series":"1.2.826.0.1.3680043.8.498.12841613654261402794761304276169172751","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11520508017843738484108845144418061602","series":"1.2.826.0.1.3680043.8.498.48580422696516156704071547885311407971","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388","series":"1.2.826.0.1.3680043.8.498.88668748948719835841370360529035928107","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11533971444023236216745801026152089075","series":"1.2.826.0.1.3680043.8.498.10920382066225285062115526574002474943","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341","series":"1.2.826.0.1.3680043.8.498.46494884964239416726457356700536210957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11547192037231064935759879673500408149","series":"1.2.826.0.1.3680043.8.498.53477775631504123867430384868070061643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11551347885357242022697196552822735795","series":"1.2.826.0.1.3680043.8.498.12184191122612666296097547329031966103","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840","series":"1.2.826.0.1.3680043.8.498.74490634866067804318496348509978724860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425","series":"1.2.826.0.1.3680043.8.498.56690231937323552803969667069961926758","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686","series":"1.2.826.0.1.3680043.8.498.46057677617571071401088006685039187967","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.17217908849136537290448622733260703778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11586616391541918134110642641868079265","series":"1.2.826.0.1.3680043.8.498.63821432114222470816353609663815984830","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515","series":"1.2.826.0.1.3680043.8.498.91168383921584476059699615410102704423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997","series":"1.2.826.0.1.3680043.8.498.10788714070459964231154884978901427160","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202","series":"1.2.826.0.1.3680043.8.498.26211889083931134137723504252837718908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860","series":"1.2.826.0.1.3680043.8.498.13174573027705841621612196093937470724","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11605266731872772432220415835172557961","series":"1.2.826.0.1.3680043.8.498.11072497440697635483247824405773164716","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874","series":"1.2.826.0.1.3680043.8.498.49076927248449903008886900037764768627","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","series":"1.2.826.0.1.3680043.8.498.77043320464734359347687094390195619494","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612","series":"1.2.826.0.1.3680043.8.498.82082186978650944031753201497309408809","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014","series":"1.2.826.0.1.3680043.8.498.56980918400156234286200431831739238744","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181","series":"1.2.826.0.1.3680043.8.498.59907782726032629195011270876548111559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11654064547593365398103107850228629066","series":"1.2.826.0.1.3680043.8.498.66479961893029036764451304937936427697","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493","series":"1.2.826.0.1.3680043.8.498.61470548704655974366012651514173599235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11674925658184069242322832524274428408","series":"1.2.826.0.1.3680043.8.498.92389345774865285380010592448939734905","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11679981444785137570894759919380932611","series":"1.2.826.0.1.3680043.8.498.42445699325606576438675450493250967077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835","series":"1.2.826.0.1.3680043.8.498.53115654595278075315958373520837538927","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.11721112507011653246778407510825259986","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917","series":"1.2.826.0.1.3680043.8.498.98363103289337292053305043824166173303","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429","series":"1.2.826.0.1.3680043.8.498.84690185969250404025839147476820058504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.94933571072989574673458681870855373755","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374","series":"1.2.826.0.1.3680043.8.498.12969702035319550228493789883047620748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11725843001854130477287484865097535511","series":"1.2.826.0.1.3680043.8.498.13135723618235592724757253241546766753","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607","series":"1.2.826.0.1.3680043.8.498.23184087416768138430683001635568889456","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11762352496600661264881556836248251400","series":"1.2.826.0.1.3680043.8.498.69284203293964247724732103728581889980","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756","series":"1.2.826.0.1.3680043.8.498.90312542992412683157082113686184827836","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803","series":"1.2.826.0.1.3680043.8.498.22793554490195421179202308721676214800","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11785138333495400389510795613252286069","series":"1.2.826.0.1.3680043.8.498.17113452382127112018244866712873772030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","series":"1.2.826.0.1.3680043.8.498.12246886798679468811733288571761884160","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033","series":"1.2.826.0.1.3680043.8.498.12423781803578209146265470032789750446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762","series":"1.2.826.0.1.3680043.8.498.11141658821914679354926570992468612604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","series":"1.2.826.0.1.3680043.8.498.59489389374059675889908456806884165681","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11807494462283790976935891042637502434","series":"1.2.826.0.1.3680043.8.498.30719585769301737582148059012292802409","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969","series":"1.2.826.0.1.3680043.8.498.13089747010640913050793290257815887034","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11836929166973549727078374306175992474","series":"1.2.826.0.1.3680043.8.498.80582839735061651292975374264947743417","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417","series":"1.2.826.0.1.3680043.8.498.92014766167558307381787224272820673730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","series":"1.2.826.0.1.3680043.8.498.12355604882573707549506011246746173691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.43180105778486656621656592937308956717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471","series":"1.2.826.0.1.3680043.8.498.93087613154659337480746028979955917841","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184","series":"1.2.826.0.1.3680043.8.498.10919451611749214187843082268149725961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794","series":"1.2.826.0.1.3680043.8.498.11602778190381867419812122399512229997","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673","series":"1.2.826.0.1.3680043.8.498.36593142142014224955326868565363288550","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672","series":"1.2.826.0.1.3680043.8.498.44561613442358206359832966240623285331","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708","series":"1.2.826.0.1.3680043.8.498.34060304028762717919914236244872692371","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.43973876237487556863208472794813969162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11919420571588879483419578078755803351","series":"1.2.826.0.1.3680043.8.498.39752409390341404876279845065981570798","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182","series":"1.2.826.0.1.3680043.8.498.10121513054086095412322965074054631327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.64358086893036637274674116716627614756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185","series":"1.2.826.0.1.3680043.8.498.17407705225351624012629206989199242672","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014","series":"1.2.826.0.1.3680043.8.498.30248526686699683044052393530122280747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769","series":"1.2.826.0.1.3680043.8.498.12574186653905857344225897523856059460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480","series":"1.2.826.0.1.3680043.8.498.88953166018176063880963523889267947838","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","series":"1.2.826.0.1.3680043.8.498.37360736441830873870609190110962299923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11979906027101464432999416052202616662","series":"1.2.826.0.1.3680043.8.498.94660774435601561379562813458656525061","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673","series":"1.2.826.0.1.3680043.8.498.10597134948033789421495482317887919826","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771","series":"1.2.826.0.1.3680043.8.498.87376458110164264621648119429843477417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473","series":"1.2.826.0.1.3680043.8.498.46353146345694575659374928367751759684","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","series":"1.2.826.0.1.3680043.8.498.79720215474764774702548415752693811767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12014959202771141042815134335571931814","series":"1.2.826.0.1.3680043.8.498.51733247994199038662971710586486776890","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122","series":"1.2.826.0.1.3680043.8.498.53994507761212596806324923831913998928","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205","series":"1.2.826.0.1.3680043.8.498.48569222386746885184434102922614029325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12041491173612341571223481616114183729","series":"1.2.826.0.1.3680043.8.498.24933948061011119327037275675401586799","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179","series":"1.2.826.0.1.3680043.8.498.13304826355926043409737018065517569403","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12063742398421783626420335095128808816","series":"1.2.826.0.1.3680043.8.498.12431315921473648516146669951664389035","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12067625141402105065337444349201280606","series":"1.2.826.0.1.3680043.8.498.53199895545316702580942439651280415057","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336","series":"1.2.826.0.1.3680043.8.498.70459438881844960296661655321704914548","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491","series":"1.2.826.0.1.3680043.8.498.85455570635861546075428779379900607944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415","series":"1.2.826.0.1.3680043.8.498.10532646787393135826644429225202855051","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051","series":"1.2.826.0.1.3680043.8.498.27624758338950476960122592322842340686","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12130605331118953242364684736005454581","series":"1.2.826.0.1.3680043.8.498.47221704273177959416320662780796940655","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12132814192991105841939278718608095213","series":"1.2.826.0.1.3680043.8.498.18591266311312655683060800151704636056","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711","series":"1.2.826.0.1.3680043.8.498.73087777177687731099762802427950701940","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.54340531841572494196144456445107072981","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457","series":"1.2.826.0.1.3680043.8.498.57816490179813235403516653981615881055","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169","series":"1.2.826.0.1.3680043.8.498.53075615107641137874529319271382048381","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12168045829469202472124733625621943469","series":"1.2.826.0.1.3680043.8.498.98675776690771869141356240669824507408","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","series":"1.2.826.0.1.3680043.8.498.10531843757009922541985350886528056132","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.98971547363766327477578333905041086202","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515","series":"1.2.826.0.1.3680043.8.498.53377742248099706625267319531893108762","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789","series":"1.2.826.0.1.3680043.8.498.44963717367832755137043256222984073592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12192455583120681639582915365754780698","series":"1.2.826.0.1.3680043.8.498.10233743899817775180073068659125375196","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.12682525940036380732605091818935114871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681","series":"1.2.826.0.1.3680043.8.498.22887331528221312893890758062604832094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986","series":"1.2.826.0.1.3680043.8.498.13042339347066750863315740638406205961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12221892384948220776502422531414659289","series":"1.2.826.0.1.3680043.8.498.96248433574585100497285536231218502364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.60103025621389962299745293089037123028","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120","series":"1.2.826.0.1.3680043.8.498.67383840232122570305852995056076986372","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837","series":"1.2.826.0.1.3680043.8.498.89039658629936993140372591293073230068","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","series":"1.2.826.0.1.3680043.8.498.11072435008739587927117532330017526788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.16652685328012622124543054841874861090","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.56769496223753360304389512240564484884","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876","series":"1.2.826.0.1.3680043.8.498.48600589356178590922880553961979229314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266054872802015665014176014323258615","series":"1.2.826.0.1.3680043.8.498.11555395012494895204669865984674237276","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039","series":"1.2.826.0.1.3680043.8.498.11816251264066016838728615748114628797","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","series":"1.2.826.0.1.3680043.8.498.82244179458701501947613155384596080837","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683","series":"1.2.826.0.1.3680043.8.498.47363125232922539916347796380503447339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.98677110094651192200915138994460120077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.13661421102001431981857407141453627040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12294073960417956629256115458135748872","series":"1.2.826.0.1.3680043.8.498.69714052603222236691650656168249978079","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937","series":"1.2.826.0.1.3680043.8.498.12918152548742831407326026602621394907","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508","series":"1.2.826.0.1.3680043.8.498.11188250230313040187691714505466669859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668","series":"1.2.826.0.1.3680043.8.498.54900075915279412841403371799181669854","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12308357850075929758995438211317707857","series":"1.2.826.0.1.3680043.8.498.83675816456081067207664886447061939540","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097","series":"1.2.826.0.1.3680043.8.498.76615661984378663889588753827208881376","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914","series":"1.2.826.0.1.3680043.8.498.10039733347836310036939398053348895446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12335573784261147388984372283481919933","series":"1.2.826.0.1.3680043.8.498.10675902318704312847451548991028571317","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12347007036329783363431172840583325937","series":"1.2.826.0.1.3680043.8.498.11603685804070669485751089768792866547","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219","series":"1.2.826.0.1.3680043.8.498.10904463693784431009629136736698250528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12352132666379687393012255589355771387","series":"1.2.826.0.1.3680043.8.498.62141733549555564581759714612960787173","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12356611369549236159671896247094661767","series":"1.2.826.0.1.3680043.8.498.11802576846782643004191732136843203394","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269","series":"1.2.826.0.1.3680043.8.498.82596488770010937500802219259403706113","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","series":"1.2.826.0.1.3680043.8.498.12002991624729783262465057339310468032","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.98386972751180337251355880451728780753","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12383716855639375235119728470042467411","series":"1.2.826.0.1.3680043.8.498.11117294182109751924112334404928666484","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387075993162655781372091860026675548","series":"1.2.826.0.1.3680043.8.498.93223307368091972718426080809500708174","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062","series":"1.2.826.0.1.3680043.8.498.27088710662583687902059306438696843494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739","series":"1.2.826.0.1.3680043.8.498.74702419628045052846195194723775141777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020","series":"1.2.826.0.1.3680043.8.498.13008949561598715999143716754772251555","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831","series":"1.2.826.0.1.3680043.8.498.10344165389179009670373467929293508166","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010","series":"1.2.826.0.1.3680043.8.498.12258720129040208740682154698860529992","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12432974885645948103184031984290883090","series":"1.2.826.0.1.3680043.8.498.44667901025473459103853743161401571081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876","series":"1.2.826.0.1.3680043.8.498.98874278180270741491125047185535193884","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945","series":"1.2.826.0.1.3680043.8.498.30578400972104371422156040149306494449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","series":"1.2.826.0.1.3680043.8.498.52948101151039326316710854459332108696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.10662999273921684120276899114140417448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084","series":"1.2.826.0.1.3680043.8.498.78837692422695491042173266705970505002","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12487362549907644519947672428024977201","series":"1.2.826.0.1.3680043.8.498.90856831199844983560155405263849549235","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642","series":"1.2.826.0.1.3680043.8.498.15582840249152689203027540003622274388","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951","series":"1.2.826.0.1.3680043.8.498.34968571761542232265504487551235916080","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072","series":"1.2.826.0.1.3680043.8.498.44148756552717723274171021004123516500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919","series":"1.2.826.0.1.3680043.8.498.11257197106770766228713602585969068308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.64009054822190539876510980455349282019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12510856755988712246013743789183840748","series":"1.2.826.0.1.3680043.8.498.58808748381679647886717248321966308990","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12518297594498173981937681848554641851","series":"1.2.826.0.1.3680043.8.498.51210166181139095673054571177832306363","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12523676732876902778577970655429060858","series":"1.2.826.0.1.3680043.8.498.13825475917904783851393966449448266732","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158","series":"1.2.826.0.1.3680043.8.498.79904843162157186498640317278639366544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494","series":"1.2.826.0.1.3680043.8.498.68911129473791149367942332050373607252","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643","series":"1.2.826.0.1.3680043.8.498.22322674601261697612789169004998225302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988","series":"1.2.826.0.1.3680043.8.498.53814263143818363025637262218935485767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727","series":"1.2.826.0.1.3680043.8.498.12020309505544786732824972594452875375","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616","series":"1.2.826.0.1.3680043.8.498.55100336585957516713967783944316571309","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.11890117362879911392838632929938487214","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292","series":"1.2.826.0.1.3680043.8.498.12834697336291310348055640120899708794","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298","series":"1.2.826.0.1.3680043.8.498.29048745169190458326746294475957425229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468","series":"1.2.826.0.1.3680043.8.498.11396537222638624188257089493757656177","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691","series":"1.2.826.0.1.3680043.8.498.97990363686186146579545673010033986066","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159","series":"1.2.826.0.1.3680043.8.498.12650167130655713645807960475222818047","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584","series":"1.2.826.0.1.3680043.8.498.10308941073922061910558687985333192395","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702","series":"1.2.826.0.1.3680043.8.498.86900867163224489952492053772704695220","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442","series":"1.2.826.0.1.3680043.8.498.11805943007256167834632366535666601945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.52910243662463919938452237985885804966","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491","series":"1.2.826.0.1.3680043.8.498.12768502649972655865937038853130362216","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12642641661988102031826128876502303807","series":"1.2.826.0.1.3680043.8.498.90803212993235480595119398552433296041","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12647791319443598318497312837147189300","series":"1.2.826.0.1.3680043.8.498.46431565831193900847102779290086875809","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625","series":"1.2.826.0.1.3680043.8.498.43229679083468404333329265009590600037","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220","series":"1.2.826.0.1.3680043.8.498.29554687101728108233947237265383503753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12674930973831274527829261576087154644","series":"1.2.826.0.1.3680043.8.498.10522161291344094509851110825643020189","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12688954493547892047021455671738310186","series":"1.2.826.0.1.3680043.8.498.21597155411239550539205836828778999670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.11118195156228185257091646709817623422","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111","series":"1.2.826.0.1.3680043.8.498.67644112337807971365514928309118426499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803","series":"1.2.826.0.1.3680043.8.498.70366148449917526312348064320058888482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12709854414521475738435319009767427434","series":"1.2.826.0.1.3680043.8.498.12630927899696625637294150316045078778","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950","series":"1.2.826.0.1.3680043.8.498.38160282861019742515978524138274553793","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231","series":"1.2.826.0.1.3680043.8.498.11940491742101567581202315459134947995","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405","series":"1.2.826.0.1.3680043.8.498.95735811588737819484848613659973683432","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122","series":"1.2.826.0.1.3680043.8.498.44474088976405934873947480937472128372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492","series":"1.2.826.0.1.3680043.8.498.10382469984093090255308613282626537567","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12759704450656741777120047828560448622","series":"1.2.826.0.1.3680043.8.498.96801216082467563278291848291863109721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12762076104929080571464647825161185643","series":"1.2.826.0.1.3680043.8.498.71754470068528497094652603813197023143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908","series":"1.2.826.0.1.3680043.8.498.39068258907727215442945648496990758499","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.25302986085127885822753226806459036993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333","series":"1.2.826.0.1.3680043.8.498.72362083906157700696449271734427077176","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180","series":"1.2.826.0.1.3680043.8.498.58585973062424329989836859688003798765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532","series":"1.2.826.0.1.3680043.8.498.98831565753675240739609409848860987107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.10896891967508619885496049345358439491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492","series":"1.2.826.0.1.3680043.8.498.48113798985258668035677850099478338495","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151","series":"1.2.826.0.1.3680043.8.498.46859949219294215565893588392219581454","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996","series":"1.2.826.0.1.3680043.8.498.13011846826267892071360068692653828009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046","series":"1.2.826.0.1.3680043.8.498.10449985389044429296583643754378086478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751","series":"1.2.826.0.1.3680043.8.498.88882823705510684044863839299863241471","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695","series":"1.2.826.0.1.3680043.8.498.79733546647407765422835954383620187109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12837030643153472711780894512565629507","series":"1.2.826.0.1.3680043.8.498.73144055211395221931634254784015530230","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404","series":"1.2.826.0.1.3680043.8.498.18814547003640332231903118075290349152","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655","series":"1.2.826.0.1.3680043.8.498.64995242626259109554220979243736010087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","series":"1.2.826.0.1.3680043.8.498.97188444912164298855416230650877390313","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.62024837984566674745838214105863635337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.28525190273943248451286603859501686527","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.12337656742326571576598459960057461199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12942677240800343332600884989364651588","series":"1.2.826.0.1.3680043.8.498.21172001710441200566654831356046621353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544","series":"1.2.826.0.1.3680043.8.498.54827871237190403618308006316676553690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","series":"1.2.826.0.1.3680043.8.498.32515971935290467596850586267878851318","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.19539108257598770084498890598864085375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626","series":"1.2.826.0.1.3680043.8.498.12559140215338339478264081093893384367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285","series":"1.2.826.0.1.3680043.8.498.22992620357884149661482526453114678956","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.10577210597786937162496073412182909868","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172","series":"1.2.826.0.1.3680043.8.498.77016089810283152365286574293162432511","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12986517636774565418070546261705439006","series":"1.2.826.0.1.3680043.8.498.34246654630903438827659434996097395068","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569","series":"1.2.826.0.1.3680043.8.498.49143343504220549424411207320491399482","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285","series":"1.2.826.0.1.3680043.8.498.76025500050383741576882539141374106705","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088","series":"1.2.826.0.1.3680043.8.498.90527762083320810616016835345058142499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13022585036130454941964190427558363602","series":"1.2.826.0.1.3680043.8.498.56027277585352782785456703320523246069","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200","series":"1.2.826.0.1.3680043.8.498.12610160718809753624598529169444186643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594","series":"1.2.826.0.1.3680043.8.498.53167431832618090034014650791641168478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13028226019192925363616760332442456953","series":"1.2.826.0.1.3680043.8.498.50119255530858548928155883015497981209","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.88738672058635335006195497475417398947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13054766954092919774634499780241210835","series":"1.2.826.0.1.3680043.8.498.92044767766885613207572270223464211398","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","series":"1.2.826.0.1.3680043.8.498.16830740143443080800100376868682786168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974","series":"1.2.826.0.1.3680043.8.498.64521214120876353921274224178000134408","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13077357855769903335728068420884303993","series":"1.2.826.0.1.3680043.8.498.86251106130338853288635198759039773884","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703","series":"1.2.826.0.1.3680043.8.498.65491541387025635978670199002630265740","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742","series":"1.2.826.0.1.3680043.8.498.77593935918251886725799992273080832065","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487","series":"1.2.826.0.1.3680043.8.498.93670415844657007226358756272618534676","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456","series":"1.2.826.0.1.3680043.8.498.86488633500999378546669003020200053177","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.12500753101296511813293218651834252884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786","series":"1.2.826.0.1.3680043.8.498.13339257502915474908082602062509778843","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712","series":"1.2.826.0.1.3680043.8.498.11926086837331592971235908234554516968","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","series":"1.2.826.0.1.3680043.8.498.11373113426818167339923900634173276732","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13125910783461518266627225743585018143","series":"1.2.826.0.1.3680043.8.498.44792891978921908207337242349920408378","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048","series":"1.2.826.0.1.3680043.8.498.10154551002388705599640950484590291575","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142","series":"1.2.826.0.1.3680043.8.498.11753345365361262407552786454558778705","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258","series":"1.2.826.0.1.3680043.8.498.68030798360392281188856418194954368300","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.17112877514812485476543202203956550625","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545","series":"1.2.826.0.1.3680043.8.498.96965711659153543373280925591603523770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925","series":"1.2.826.0.1.3680043.8.498.15228187357203902055610405158930703099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775","series":"1.2.826.0.1.3680043.8.498.40062040526655603799936681561800743682","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13148805278538458401210722493048566510","series":"1.2.826.0.1.3680043.8.498.10423664825586402680118020498387884844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956","series":"1.2.826.0.1.3680043.8.498.11723417050732618048241721015605082969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312","series":"1.2.826.0.1.3680043.8.498.66086838551650010905732920293582256963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296","series":"1.2.826.0.1.3680043.8.498.53181531191234409991074042809011577941","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680","series":"1.2.826.0.1.3680043.8.498.84324351033542999050764833123963804032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205","series":"1.2.826.0.1.3680043.8.498.45998951916696391696369295663503360519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815","series":"1.2.826.0.1.3680043.8.498.59835215508728069731959283293276698507","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","series":"1.2.826.0.1.3680043.8.498.25363878782585743698864140115059170793","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13179829911816163852815103052496326166","series":"1.2.826.0.1.3680043.8.498.12117158130340035648568305526043034528","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13200225436827306542001129042739578441","series":"1.2.826.0.1.3680043.8.498.10260035364406876815719322958772444042","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.44224441873122068267896000657789592068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605","series":"1.2.826.0.1.3680043.8.498.27455412666467363291433093976152218185","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312","series":"1.2.826.0.1.3680043.8.498.10946629711257688362768337690690078986","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226","series":"1.2.826.0.1.3680043.8.498.49445657524276570361404797785820067639","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.25674611753338662533530472182915944359","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308","series":"1.2.826.0.1.3680043.8.498.52425995352921691791654659570002187778","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13256173302993031630968402868348974108","series":"1.2.826.0.1.3680043.8.498.40115217237380824531574400386759046047","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","series":"1.2.826.0.1.3680043.8.498.21082499439875857466670337482188300852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733","series":"1.2.826.0.1.3680043.8.498.53979703006362751386679294709664350996","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891","series":"1.2.826.0.1.3680043.8.498.60329250734334275308110817172897714281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320","series":"1.2.826.0.1.3680043.8.498.50801011989962985107663124556992933028","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634","series":"1.2.826.0.1.3680043.8.498.27486575750905262294770025418222186006","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.21645253967107068689450581612579844855","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.12815869235381419255690573555101639618","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.45689743229914325224944955050735546295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13314276151860609710431717811242659651","series":"1.2.826.0.1.3680043.8.498.10031285995955397064012296381020595912","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644","series":"1.2.826.0.1.3680043.8.498.11780476638708019510962986716883021867","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13330694332190836651711643932953274573","series":"1.2.826.0.1.3680043.8.498.68006432551884096599533371997862976039","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13335786652653204791740906118815827207","series":"1.2.826.0.1.3680043.8.498.78929304240184330361687298309208911359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879","series":"1.2.826.0.1.3680043.8.498.12455429767246432237380297733395933603","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170","series":"1.2.826.0.1.3680043.8.498.48086020295913907140202526688567389228","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13361877959630120404875545203408927433","series":"1.2.826.0.1.3680043.8.498.13068826285388310116789644808220557893","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13367691581890020433564928882602648123","series":"1.2.826.0.1.3680043.8.498.85433020514631650039110095305530853912","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388","series":"1.2.826.0.1.3680043.8.498.83124362455493568704826636328992665321","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.17547497673960695811629174516009936531","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194","series":"1.2.826.0.1.3680043.8.498.17656711803599248377594417280838418867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680","series":"1.2.826.0.1.3680043.8.498.11361278372846694094467798839239308576","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981","series":"1.2.826.0.1.3680043.8.498.15068180950329924417117001072707989168","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965","series":"1.2.826.0.1.3680043.8.498.12768796635320985502103512661992236472","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933","series":"1.2.826.0.1.3680043.8.498.13142794401056816483948480640810903137","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.47123395599170166434705820763512813026","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.16023888630574555554946417102233270325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636","series":"1.2.826.0.1.3680043.8.498.12668030742047794443569918056877292188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235","series":"1.2.826.0.1.3680043.8.498.12433806266505776793883928774389850982","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969","series":"1.2.826.0.1.3680043.8.498.84248772337190968640619176901758841181","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084","series":"1.2.826.0.1.3680043.8.498.12447356330970976484925653534860110100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13958765761660666865188384031150827763","series":"1.2.826.0.1.3680043.8.498.11591009668867802515909576049868983567","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978","series":"1.2.826.0.1.3680043.8.498.26464637123231949995447814938231603743","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446","series":"1.2.826.0.1.3680043.8.498.11642976855722962339571747919854290104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14137431039800236510749401303688858447","series":"1.2.826.0.1.3680043.8.498.56021007445920340133288476494603049504","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14218113768651197803980865495163189825","series":"1.2.826.0.1.3680043.8.498.63445552377847551716566741147492668736","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759","series":"1.2.826.0.1.3680043.8.498.12744329721348118012531615448809784040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","series":"1.2.826.0.1.3680043.8.498.13204302639611317127353024933529432138","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532","series":"1.2.826.0.1.3680043.8.498.64683873754721161320182343190417653716","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.47697965570206304531998487382497838733","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557","series":"1.2.826.0.1.3680043.8.498.73151606215530194263189921650078829272","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211","series":"1.2.826.0.1.3680043.8.498.77238395454969335032379249543235291603","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735","series":"1.2.826.0.1.3680043.8.498.38558604628961050502187561748184601425","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14776241668061173271058268311868302014","series":"1.2.826.0.1.3680043.8.498.77116625450463802481048663732946466960","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770","series":"1.2.826.0.1.3680043.8.498.82458383001341640560189543004154845642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14959561028578335448289993045623217861","series":"1.2.826.0.1.3680043.8.498.47661633853719337822594359422023972834","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069","series":"1.2.826.0.1.3680043.8.498.84598434548080726955381343448453362304","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.89105738709335100634649688811562086779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755","series":"1.2.826.0.1.3680043.8.498.97864612188859501297474829638395317493","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071","series":"1.2.826.0.1.3680043.8.498.70736644965187936782877063779653141131","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15326951535563258414956521742264374802","series":"1.2.826.0.1.3680043.8.498.17686472396988866378012308938108054575","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.12328220757960100225176727103072589880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351","series":"1.2.826.0.1.3680043.8.498.13058494097826384201690488548902611897","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269","series":"1.2.826.0.1.3680043.8.498.96209010301828647894769821695346562942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397","series":"1.2.826.0.1.3680043.8.498.53902919065122512906120077458438676448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865","series":"1.2.826.0.1.3680043.8.498.31795652286153805948478791096843053535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742","series":"1.2.826.0.1.3680043.8.498.87952262721016936036376345530006899422","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15851463019383290210760064201595021029","series":"1.2.826.0.1.3680043.8.498.24738992440574745230565101194140601082","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764","series":"1.2.826.0.1.3680043.8.498.37061323368096484508870341984599174916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.56489188975219855943731152460524612324","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","series":"1.2.826.0.1.3680043.8.498.18812668041348918806800136283080472143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921","series":"1.2.826.0.1.3680043.8.498.10273917972418396700537785096040650198","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16258190393569006141285550986533848849","series":"1.2.826.0.1.3680043.8.498.45489677931740970620591677553743119481","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16364873390453802195568603605191835957","series":"1.2.826.0.1.3680043.8.498.11316867052141580674821636226734622872","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331","series":"1.2.826.0.1.3680043.8.498.12106594461614209740814162844238669020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16527747624688737126118546487228060848","series":"1.2.826.0.1.3680043.8.498.94289114197515144799389071375783987500","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16538738179840493095338324009830673843","series":"1.2.826.0.1.3680043.8.498.12774720085357708695885978115998806722","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350","series":"1.2.826.0.1.3680043.8.498.10227133879988062044388617501111888601","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849","series":"1.2.826.0.1.3680043.8.498.10348994694194643860284202476568088061","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16709565360569239280032743619008187278","series":"1.2.826.0.1.3680043.8.498.10578186373622208964955220152320493041","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16724013341359693140331158148551346369","series":"1.2.826.0.1.3680043.8.498.25592896488543437051375167420950620443","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.52588485467989835018516090707006064483","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16862596867845295946732698348903803700","series":"1.2.826.0.1.3680043.8.498.52309418953053354621227316332603033667","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16886658450293145802897654768506439494","series":"1.2.826.0.1.3680043.8.498.17050129296690074611565245342312193745","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835","series":"1.2.826.0.1.3680043.8.498.10450132118492513079615987759298999272","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.79186759514951470066801190764713246607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707","series":"1.2.826.0.1.3680043.8.498.10547267834933929410782514723294988622","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17160610480070701553118850947260201608","series":"1.2.826.0.1.3680043.8.498.70698264748615562776670758094314585903","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876","series":"1.2.826.0.1.3680043.8.498.34429381565245359369969302555969879610","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823","series":"1.2.826.0.1.3680043.8.498.83617091922366426069393280086650005805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17420673628956367425739018933968443620","series":"1.2.826.0.1.3680043.8.498.31921790475319649674194502669447688409","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17547781127981007962008163248942714330","series":"1.2.826.0.1.3680043.8.498.18736504215983219410522103979884358100","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319","series":"1.2.826.0.1.3680043.8.498.58434720010080671651531566651388473046","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433","series":"1.2.826.0.1.3680043.8.498.13289971568085285202641440452640579046","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.66328090334478063137101654819070723636","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885","series":"1.2.826.0.1.3680043.8.498.35108987562543000396562068226962838786","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916","series":"1.2.826.0.1.3680043.8.498.76330770589565894445338637399798918180","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728","series":"1.2.826.0.1.3680043.8.498.98176829716228442349312993155047165281","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730","series":"1.2.826.0.1.3680043.8.498.81001123812469341354147972738734177063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043","series":"1.2.826.0.1.3680043.8.498.26629115744237555658227437133065519815","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285","series":"1.2.826.0.1.3680043.8.498.11777138406724168392723272190474804401","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584","series":"1.2.826.0.1.3680043.8.498.11526669257578654115956099838593623802","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124","series":"1.2.826.0.1.3680043.8.498.10762203843506972411800239191793929665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067","series":"1.2.826.0.1.3680043.8.498.40852876956618878978950616342252218048","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378","series":"1.2.826.0.1.3680043.8.498.54971859142466406163844002076964794013","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18518393141873346710197645305647124682","series":"1.2.826.0.1.3680043.8.498.39328132517954807520571275882285831570","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570","series":"1.2.826.0.1.3680043.8.498.51798411213634147240927433815919904753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","series":"1.2.826.0.1.3680043.8.498.12890391061013269471612370314798133088","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006","series":"1.2.826.0.1.3680043.8.498.11749736875502647655661426095860103249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622","series":"1.2.826.0.1.3680043.8.498.11114628041009876515045780532195647240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.33065156223690041623895456129871150464","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716","series":"1.2.826.0.1.3680043.8.498.12699192245877117948805640172811277651","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797","series":"1.2.826.0.1.3680043.8.498.11018630505146534711389302050087002039","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874","series":"1.2.826.0.1.3680043.8.498.43391145111015719774927053593476476226","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480","series":"1.2.826.0.1.3680043.8.498.10064844036614751208433717781482724643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548","series":"1.2.826.0.1.3680043.8.498.10101237742481135035557344187439703337","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358","series":"1.2.826.0.1.3680043.8.498.13230956512357507492284164573171043729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225","series":"1.2.826.0.1.3680043.8.498.42599103979617142586305032761871881660","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.19419878169415950073207624537163068568","series":"1.2.826.0.1.3680043.8.498.12793733262118172862846611445762143588","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359","series":"1.2.826.0.1.3680043.8.498.35254854011115494145906572961159895859","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.91878618850101117240256101717136256336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296","series":"1.2.826.0.1.3680043.8.498.12293483374036388293034691836878090642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310","series":"1.2.826.0.1.3680043.8.498.11902899604612320626549760552488574645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955","series":"1.2.826.0.1.3680043.8.498.99610843995157325466722641496744853850","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.29448896805415956194464239028139528518","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760","series":"1.2.826.0.1.3680043.8.498.32299184116117561858268025387624633540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948","series":"1.2.826.0.1.3680043.8.498.12544385607029900595265488105715562050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19977273190231806393528180968361858122","series":"1.2.826.0.1.3680043.8.498.29910057985434548550826049558338532646","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.12172494624534065850930467525454219800","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20156361590688072955579948030116579099","series":"1.2.826.0.1.3680043.8.498.10757890059717982918679785653259401400","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100","series":"1.2.826.0.1.3680043.8.498.11180470511504280971938836567073730120","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284","series":"1.2.826.0.1.3680043.8.498.30010517423214921969566790288116711174","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350","series":"1.2.826.0.1.3680043.8.498.71833447299661736762185464825955697967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20427402498958253193571475926580142499","series":"1.2.826.0.1.3680043.8.498.73910523705385109194833501340031315523","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.25365131802608240098247892700835109675","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.14171783791468747647708224152660582933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816","series":"1.2.826.0.1.3680043.8.498.44032355019019578078051122155154587705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20690227395174139480327772367312879333","series":"1.2.826.0.1.3680043.8.498.12224366161595988089889618174216885218","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","series":"1.2.826.0.1.3680043.8.498.50620985065038566994907711351852463271","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20850785960533583051456342090773230767","series":"1.2.826.0.1.3680043.8.498.11667304420050231192912073898166073462","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","series":"1.2.826.0.1.3680043.8.498.23635045597451958088492882101934120565","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615","series":"1.2.826.0.1.3680043.8.498.11492454834582224002828972284792096730","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199","series":"1.2.826.0.1.3680043.8.498.66475728413155039241697114405814715672","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615","series":"1.2.826.0.1.3680043.8.498.34794618411863081838860997718241080980","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203","series":"1.2.826.0.1.3680043.8.498.10074117786567575613976123424082837726","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596","series":"1.2.826.0.1.3680043.8.498.84162997494458727744760972337250614906","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21403015320378798778538823075316902483","series":"1.2.826.0.1.3680043.8.498.67076132164116720465535985796401302936","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958","series":"1.2.826.0.1.3680043.8.498.67840665032607043299627393462017340437","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21556073772857867149576001337847113661","series":"1.2.826.0.1.3680043.8.498.24733102940695063235760361823100932422","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.12116409567314195410921370804153769386","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053","series":"1.2.826.0.1.3680043.8.498.26132447334749818159383392913640315909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137","series":"1.2.826.0.1.3680043.8.498.37324849636647662864138201051696470282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","series":"1.2.826.0.1.3680043.8.498.32847377737500438050748858530419381315","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596","series":"1.2.826.0.1.3680043.8.498.24617044784380768848435672354840618687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762","series":"1.2.826.0.1.3680043.8.498.56573105307917954468859480782459581594","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","series":"1.2.826.0.1.3680043.8.498.89091739085814481696795551144656138780","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22053727963380087706894183799325067544","series":"1.2.826.0.1.3680043.8.498.36490334355074525444676423225810952320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654","series":"1.2.826.0.1.3680043.8.498.21079636721568781952508236209729887214","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.12642689937308707950933301984030640737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22137222939487357840166329770642627189","series":"1.2.826.0.1.3680043.8.498.76156143298318526227716317404703912949","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643","series":"1.2.826.0.1.3680043.8.498.11809232047725985614296913945118662040","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013","series":"1.2.826.0.1.3680043.8.498.38010227306414722961708883626363136998","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005","series":"1.2.826.0.1.3680043.8.498.13111656775499769414585826572445331952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060","series":"1.2.826.0.1.3680043.8.498.11247611492942014938450299640569190705","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644","series":"1.2.826.0.1.3680043.8.498.34491328021323724703034269313224125087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22594742481654038247097269358877852869","series":"1.2.826.0.1.3680043.8.498.11653103196658098991894329022975854039","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22617494262793136174349268379191631799","series":"1.2.826.0.1.3680043.8.498.13258986301845935206213356218080710595","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550","series":"1.2.826.0.1.3680043.8.498.29708722375286583654926491304915360363","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.11012066421729811002231381529887461526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817","series":"1.2.826.0.1.3680043.8.498.77665664207808174430151013729024950069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","series":"1.2.826.0.1.3680043.8.498.45428601233787588940719158936408241733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091","series":"1.2.826.0.1.3680043.8.498.50909861836097068869469977095576412304","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980","series":"1.2.826.0.1.3680043.8.498.15040432130556352301352594899163457524","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744","series":"1.2.826.0.1.3680043.8.498.44508787788899483621498814438839746929","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738","series":"1.2.826.0.1.3680043.8.498.42699407535969811569714073845904884276","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.12789026972720181021327531822855395162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23518857399095666263158269685495375084","series":"1.2.826.0.1.3680043.8.498.37189816932667406411856861222288598398","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","series":"1.2.826.0.1.3680043.8.498.61295311345824665267078009784940978423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23713320796530316370039202981138376061","series":"1.2.826.0.1.3680043.8.498.69316538671700586797863994429414069013","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23757007145576852496599816072431075057","series":"1.2.826.0.1.3680043.8.498.13892146040755469094852463275316237298","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23833933822461809237193998300076847210","series":"1.2.826.0.1.3680043.8.498.11638044452207504541300864787718767327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277","series":"1.2.826.0.1.3680043.8.498.11675742812263873810174068316154454848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413","series":"1.2.826.0.1.3680043.8.498.60145947634368129966688453668381887930","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916","series":"1.2.826.0.1.3680043.8.498.13118640091992688090338434370219225172","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200","series":"1.2.826.0.1.3680043.8.498.13028520037296832209340842331503827475","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631","series":"1.2.826.0.1.3680043.8.498.12338193034184948509244073689011265947","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504","series":"1.2.826.0.1.3680043.8.498.98481558226317346076716722829785918396","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24369332907731805266450307536793220148","series":"1.2.826.0.1.3680043.8.498.29265242399891341093351675303514899706","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614","series":"1.2.826.0.1.3680043.8.498.42833313646351121965321396459892056591","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314","series":"1.2.826.0.1.3680043.8.498.11236109009719282400843886712346073372","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24635589616467409099788838058892886889","series":"1.2.826.0.1.3680043.8.498.12435710986410276696667828631756536493","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593","series":"1.2.826.0.1.3680043.8.498.17817027685839994698575171031150160808","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","series":"1.2.826.0.1.3680043.8.498.40706330670948683248288888701250818672","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720","series":"1.2.826.0.1.3680043.8.498.68661421328957860040688364216963100029","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187","series":"1.2.826.0.1.3680043.8.498.11791116892739648845869424239781280321","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935","series":"1.2.826.0.1.3680043.8.498.11524537723583674949081629148524922688","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861","series":"1.2.826.0.1.3680043.8.498.29757090984929419949146881558446820857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.12354753594609904252041979450256296318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25188196199869378521949969245728443004","series":"1.2.826.0.1.3680043.8.498.48302518121791655153369201262648066168","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","series":"1.2.826.0.1.3680043.8.498.28519868389452758272892187097882501222","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810","series":"1.2.826.0.1.3680043.8.498.13212979279483302931276369266073797700","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25394815405242666778213609728680620077","series":"1.2.826.0.1.3680043.8.498.53556620699127132781861804288375569293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25440342930186170531037102906687829898","series":"1.2.826.0.1.3680043.8.498.39584235693199270766929604824455073858","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863","series":"1.2.826.0.1.3680043.8.498.10793120247086431655974099718632673003","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882","series":"1.2.826.0.1.3680043.8.498.12771549602237763789841828877947665531","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379","series":"1.2.826.0.1.3680043.8.498.97751683836630868446089922763345084240","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505","series":"1.2.826.0.1.3680043.8.498.74690453620208178087094935167596492240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","series":"1.2.826.0.1.3680043.8.498.69424242864603587968136288315964371466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.10431359907759302112904561594331019770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103","series":"1.2.826.0.1.3680043.8.498.89939501324449389946320012812141364608","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405","series":"1.2.826.0.1.3680043.8.498.58497436069528797502046636425964222241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260","series":"1.2.826.0.1.3680043.8.498.48091815231885820195492095956646344826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25856396695552621816673203639618612020","series":"1.2.826.0.1.3680043.8.498.19890616605936060801160044457302425019","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25909078145370189195643713208718315003","series":"1.2.826.0.1.3680043.8.498.71772712412964624051113584866476501880","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036","series":"1.2.826.0.1.3680043.8.498.46586703046781701722914118074283038723","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547","series":"1.2.826.0.1.3680043.8.498.52075218526162058729744163345966846369","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604","series":"1.2.826.0.1.3680043.8.498.37787725727928759720580165934090952463","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568","series":"1.2.826.0.1.3680043.8.498.15509168988352948201576488538374016031","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705","series":"1.2.826.0.1.3680043.8.498.11453327356994747436277817171909225360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26333849754345698117396119042432557445","series":"1.2.826.0.1.3680043.8.498.64398940143800291032140475284456439065","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.68075325317212163269347009866376511139","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691","series":"1.2.826.0.1.3680043.8.498.10734534740204958355383287320420072850","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26512333939158074470511885252122191522","series":"1.2.826.0.1.3680043.8.498.50806715637563419819950458395996753325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","series":"1.2.826.0.1.3680043.8.498.94021548006860003414087462984340082468","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.42079706488655999241887378983689047215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26792665054823243997838322477587733719","series":"1.2.826.0.1.3680043.8.498.58247928682956119033091604697458955488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694","series":"1.2.826.0.1.3680043.8.498.17374658499156206279245260881313123805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.45994383167323604703135183740637817315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26989584479346775218799121753035575730","series":"1.2.826.0.1.3680043.8.498.44373730510978665887627354331401411697","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661","series":"1.2.826.0.1.3680043.8.498.11733174050861066111193631474871250512","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863","series":"1.2.826.0.1.3680043.8.498.18996178763867662189603156012044440652","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.12605449158214266473904999297796168729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27191574009501966309382307682020135126","series":"1.2.826.0.1.3680043.8.498.12529283381192391292657683666343283510","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926","series":"1.2.826.0.1.3680043.8.498.24247397768146614255637685432906501412","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","series":"1.2.826.0.1.3680043.8.498.15210676807559751185485656739364535498","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.12715211551947485275927848748971469433","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486","series":"1.2.826.0.1.3680043.8.498.23253025209189662371265024877116033468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27422742748182689233651649793019570511","series":"1.2.826.0.1.3680043.8.498.16088601401847679112156385593642021058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27460982183399364904497832933739191826","series":"1.2.826.0.1.3680043.8.498.91532103960764341093428017066808983609","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27593710915588175306316595263625481601","series":"1.2.826.0.1.3680043.8.498.50382064728635257400036947348364638691","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375","series":"1.2.826.0.1.3680043.8.498.85505040373154956189809388296568698498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780","series":"1.2.826.0.1.3680043.8.498.36863650637382162014788829203970778683","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.75965197563925741625221726837516783109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723573213501259053792522608339812759","series":"1.2.826.0.1.3680043.8.498.11829648284702890839262701698857506221","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27793455627426963285485957211105588905","series":"1.2.826.0.1.3680043.8.498.11988034402635911286299008683445658558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316","series":"1.2.826.0.1.3680043.8.498.11258616468380709778286377559672214003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","series":"1.2.826.0.1.3680043.8.498.29559271439143682801720603797532811997","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227","series":"1.2.826.0.1.3680043.8.498.10784703628844798713609543815986428779","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563","series":"1.2.826.0.1.3680043.8.498.11499393834465867861409844009251079849","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28040645186226259590182363324146810031","series":"1.2.826.0.1.3680043.8.498.10567661965201124818512395204112961401","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241","series":"1.2.826.0.1.3680043.8.498.11205761527676259962659276430211377208","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475","series":"1.2.826.0.1.3680043.8.498.11858328475198032741362701797811211913","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882","series":"1.2.826.0.1.3680043.8.498.51556316939821658963895847613678817656","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798","series":"1.2.826.0.1.3680043.8.498.36269621175613227903639795257662031015","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306","series":"1.2.826.0.1.3680043.8.498.54806612110559733802587847747311225541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098","series":"1.2.826.0.1.3680043.8.498.11710204764388899404815734499128270284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.13082869568041483457214571001992275753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28579372595938810820078670002733005121","series":"1.2.826.0.1.3680043.8.498.13328014367750779264633070828714085682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705","series":"1.2.826.0.1.3680043.8.498.72834092173477918786931438541704678040","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157","series":"1.2.826.0.1.3680043.8.498.25240380238174415037168545539287264204","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28655385212214986812679045835025619123","series":"1.2.826.0.1.3680043.8.498.60227365849335900772722117333632900502","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456","series":"1.2.826.0.1.3680043.8.498.84039187072064112678393261336996911753","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28770594655815108010532953376271504596","series":"1.2.826.0.1.3680043.8.498.12645003300366951046711539390975019146","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918","series":"1.2.826.0.1.3680043.8.498.12522910947249306845305325100739726511","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608","series":"1.2.826.0.1.3680043.8.498.75206601760489455412268202126310236834","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28951331578236330522646806432827370713","series":"1.2.826.0.1.3680043.8.498.26417155334927457491675477088721211429","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882","series":"1.2.826.0.1.3680043.8.498.45633745508258340793769769465674621363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944","series":"1.2.826.0.1.3680043.8.498.10170545250088782952611958629261159931","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586","series":"1.2.826.0.1.3680043.8.498.33952322388052600339153187816627963231","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","series":"1.2.826.0.1.3680043.8.498.11387144052728381268733197772531059805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.78228469374702730500453648205629824747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370","series":"1.2.826.0.1.3680043.8.498.12457352365186449429319512120019176158","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.12172716706688803307920696109196387240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950","series":"1.2.826.0.1.3680043.8.498.27740723261507807789247432903540632100","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.29741673719687535294395668150749894623","series":"1.2.826.0.1.3680043.8.498.26373789448072121027210910064542709128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","series":"1.2.826.0.1.3680043.8.498.10582767282153759006652720100204909285","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049","series":"1.2.826.0.1.3680043.8.498.46746500918331605111456197049362520287","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29976611818313906970687136815346573471","series":"1.2.826.0.1.3680043.8.498.42932147256719782877680209345221785621","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30104491808697888642424160917715722238","series":"1.2.826.0.1.3680043.8.498.41426980043390000091349109780142526116","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.12153095665930544560501292120515504483","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388","series":"1.2.826.0.1.3680043.8.498.12347884530934128322317182103429875762","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30233578499383083757206211752001832047","series":"1.2.826.0.1.3680043.8.498.67057646394980283253404629359472255850","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600","series":"1.2.826.0.1.3680043.8.498.37506324766169923490859273540289152730","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","series":"1.2.826.0.1.3680043.8.498.12839721100603688347292415876204546549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540","series":"1.2.826.0.1.3680043.8.498.17658861121119089844526401412496983852","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.12392133833947198191848529542568039442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30517381653283520211188050362394799510","series":"1.2.826.0.1.3680043.8.498.14440873787056589653229823177338059731","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30537361712821464927469618193893850465","series":"1.2.826.0.1.3680043.8.498.12800216888969143935005355308001916348","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223","series":"1.2.826.0.1.3680043.8.498.35321400876892812801678626919180160706","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949","series":"1.2.826.0.1.3680043.8.498.11417693418637809272983496139532620507","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520","series":"1.2.826.0.1.3680043.8.498.11709270433844573882595360382879986400","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316","series":"1.2.826.0.1.3680043.8.498.11342922748741478176962899873174776306","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30836611607735372299752226690388090391","series":"1.2.826.0.1.3680043.8.498.34936880012390754150746038003992209321","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","series":"1.2.826.0.1.3680043.8.498.10324178443663392447648740746117661057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066","series":"1.2.826.0.1.3680043.8.498.11823130856919630622164172969779714622","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","series":"1.2.826.0.1.3680043.8.498.71297256764801161492216378650732380502","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701","series":"1.2.826.0.1.3680043.8.498.71680497536735939110506583202121495453","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31137162827930164367134811210741362416","series":"1.2.826.0.1.3680043.8.498.13390771391295293181997377945234212239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","series":"1.2.826.0.1.3680043.8.498.78195115667874515127174762793539729784","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.11349013956405810062164261077153536624","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119","series":"1.2.826.0.1.3680043.8.498.48408256092890792791746616731057052860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805","series":"1.2.826.0.1.3680043.8.498.90695486554854181378756998091106244682","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486","series":"1.2.826.0.1.3680043.8.498.29161152004896481549207452658084389938","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31482725509752510642857072190147133461","series":"1.2.826.0.1.3680043.8.498.12291038127728457437726664797267778685","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914","series":"1.2.826.0.1.3680043.8.498.70625281670585984092064709508126634020","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979","series":"1.2.826.0.1.3680043.8.498.49857083817678339585385859428213417330","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721","series":"1.2.826.0.1.3680043.8.498.46714280821641537075665130613759580213","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181","series":"1.2.826.0.1.3680043.8.498.66011297307024134948135965611282477658","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100","series":"1.2.826.0.1.3680043.8.498.16792939756572824239183492415287577072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024","series":"1.2.826.0.1.3680043.8.498.54515059708610466682039181905811000131","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602","series":"1.2.826.0.1.3680043.8.498.48862022446753678690722306948284116495","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.81845363596462298600837184216412830297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32241219263926672894433308808653719495","series":"1.2.826.0.1.3680043.8.498.11682084239313734801224245144741486579","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631","series":"1.2.826.0.1.3680043.8.498.19658184270787063606497668234514537943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205","series":"1.2.826.0.1.3680043.8.498.31333433951548434296414129744090483606","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","series":"1.2.826.0.1.3680043.8.498.26593846888434504329415450868929339558","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32501049887140121289143479773995481982","series":"1.2.826.0.1.3680043.8.498.11171145952597255136461853746479002333","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399","series":"1.2.826.0.1.3680043.8.498.86286075233878920685722493389502925863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","series":"1.2.826.0.1.3680043.8.498.32272594237382645810654126384151764513","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.51805278630137177032424656421198512504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","series":"1.2.826.0.1.3680043.8.498.27080646175921030806824661261681852393","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569","series":"1.2.826.0.1.3680043.8.498.40014376201341385969678211190135472880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753","series":"1.2.826.0.1.3680043.8.498.11323273634523485717366014592768730871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","series":"1.2.826.0.1.3680043.8.498.46450798611651996818921356604391112722","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300","series":"1.2.826.0.1.3680043.8.498.11386754924702077983545139334128409495","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.11663780407215559444735901554395501037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33284228284209148349668722931307588402","series":"1.2.826.0.1.3680043.8.498.11713332236977176787155775869216954885","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33324670630108603340869986205298694836","series":"1.2.826.0.1.3680043.8.498.67439660368256617710775583641406557016","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086","series":"1.2.826.0.1.3680043.8.498.36340005991080193454570434488746840436","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.29368340292805976536611392301620887778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550","series":"1.2.826.0.1.3680043.8.498.10791620373549834409527191409070657606","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941","series":"1.2.826.0.1.3680043.8.498.10277702865105186580564291385280813128","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088","series":"1.2.826.0.1.3680043.8.498.35976811383765806907456122153209654308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687","series":"1.2.826.0.1.3680043.8.498.63023250236300677385053047008122706429","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650","series":"1.2.826.0.1.3680043.8.498.22922611655330708008678766587846281413","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028","series":"1.2.826.0.1.3680043.8.498.73356733211301520459252955340108231409","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34002185870196144582348922158474418674","series":"1.2.826.0.1.3680043.8.498.61143866754044491063214040874911887827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34058295828142876023659425586740832405","series":"1.2.826.0.1.3680043.8.498.11207697927153020319962759053451553568","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","series":"1.2.826.0.1.3680043.8.498.31790951453913355194276007245431416082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941","series":"1.2.826.0.1.3680043.8.498.66805076343767847361789966344873057335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.64418367554756344022703749575767442868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34404554752241524210538769325381023947","series":"1.2.826.0.1.3680043.8.498.10333902457885891796902377791839879798","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34441662343660776323013762699640240389","series":"1.2.826.0.1.3680043.8.498.13034654484398976739258224783685450210","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868","series":"1.2.826.0.1.3680043.8.498.20213305002116019776895908410604936735","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","series":"1.2.826.0.1.3680043.8.498.12869312195258343906686717518986461178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.10816134679797633813161224274627242382","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","series":"1.2.826.0.1.3680043.8.498.10367162593660233997239753176698867967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.87244101764847209485337045242619185008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.58495393402659035444112940594786767289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34858782760614383566891001610985452165","series":"1.2.826.0.1.3680043.8.498.13216729825823904474819132443432116023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579","series":"1.2.826.0.1.3680043.8.498.38560643987634716513969999960851520429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.54731905055879305551396838528687217237","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","series":"1.2.826.0.1.3680043.8.498.11257831575902063247050000337871186980","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425","series":"1.2.826.0.1.3680043.8.498.23257161330088534765456951091678599301","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.26321586476495595675596093299881564931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35273590880523815297008790288567354883","series":"1.2.826.0.1.3680043.8.498.53959632145172352144150875120285308051","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35338198410583984337908932327787063577","series":"1.2.826.0.1.3680043.8.498.12930533122179252272046002020577085096","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.13205897929859487014781305918501057776","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212","series":"1.2.826.0.1.3680043.8.498.86630329079425427984807052759302248720","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.66943248097724726529171909641317330853","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777","series":"1.2.826.0.1.3680043.8.498.48480441665349795305559988915217208756","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944","series":"1.2.826.0.1.3680043.8.498.36401239884729678959653379995625077554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403","series":"1.2.826.0.1.3680043.8.498.96901729824132332214815586603694523865","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201","series":"1.2.826.0.1.3680043.8.498.51474710290461336989324890426431062918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35906584224533966870202042352292878668","series":"1.2.826.0.1.3680043.8.498.11360722055402765309229157252690717756","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923183633813281697521537744486885300","series":"1.2.826.0.1.3680043.8.498.29609553254615963573735202563565043052","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.83923068175349244922187998775857406397","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35965770865730278787229040370963322662","series":"1.2.826.0.1.3680043.8.498.11528870704946483008989427890337073854","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36013620117872095227314165026090752632","series":"1.2.826.0.1.3680043.8.498.42045881877372010418721687130287825781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565","series":"1.2.826.0.1.3680043.8.498.11580439583110362701638375476079312389","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246","series":"1.2.826.0.1.3680043.8.498.84812104403122978495983748930507968158","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976","series":"1.2.826.0.1.3680043.8.498.26718018167020547476865070723776273088","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198","series":"1.2.826.0.1.3680043.8.498.11430622179650794578229114950729089695","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167","series":"1.2.826.0.1.3680043.8.498.29284693883948364379482631246319645828","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.82085152802934899470894508457001637447","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035","series":"1.2.826.0.1.3680043.8.498.10826022044822518210876638314024787783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36599566036331923007586730435849419272","series":"1.2.826.0.1.3680043.8.498.12934378810049214494396945865285643577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374","series":"1.2.826.0.1.3680043.8.498.51006047018623955825535316430138692473","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985","series":"1.2.826.0.1.3680043.8.498.64768450655749775654947747055578204093","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36825274340853796392762195174336452919","series":"1.2.826.0.1.3680043.8.498.10555071060434567084354573166729290479","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193","series":"1.2.826.0.1.3680043.8.498.10515654514775180323274473927670327875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047","series":"1.2.826.0.1.3680043.8.498.36021986133669323529681057326484032827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398","series":"1.2.826.0.1.3680043.8.498.66623875748589446440888822496177496374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876","series":"1.2.826.0.1.3680043.8.498.12626148399414995981643675457969692010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","series":"1.2.826.0.1.3680043.8.498.45315597525118513085678448335589591904","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.99277170270268252935256015471527093864","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","series":"1.2.826.0.1.3680043.8.498.73365050290015987652911028048785480116","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37347132459817985668038603215490355489","series":"1.2.826.0.1.3680043.8.498.10710353955807741405826428102934883633","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683","series":"1.2.826.0.1.3680043.8.498.10917225783091709760242231737882484221","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","series":"1.2.826.0.1.3680043.8.498.10804792800840526866304375826928031479","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849","series":"1.2.826.0.1.3680043.8.498.12587011278833911912411038763701457229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37485075271886447709158414670394650122","series":"1.2.826.0.1.3680043.8.498.74299385694069271701942363390919399297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271","series":"1.2.826.0.1.3680043.8.498.86141039281409001625389065597778168601","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023","series":"1.2.826.0.1.3680043.8.498.12903067007096762977112463241573749437","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522","series":"1.2.826.0.1.3680043.8.498.11240869747790197788792326284228107557","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654","series":"1.2.826.0.1.3680043.8.498.44295752969855253715334304079271206794","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.12957847083868470102811350999007750093","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680","series":"1.2.826.0.1.3680043.8.498.33502970026548879077669041950507260816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122","series":"1.2.826.0.1.3680043.8.498.80314729975419790634260871582982002327","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37885316830093262977685802809269466252","series":"1.2.826.0.1.3680043.8.498.31881615969289262738893482022722050122","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37969790195791584440906553207150666923","series":"1.2.826.0.1.3680043.8.498.91677561447352940052451618216433789453","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38013372055010421273425628837282557767","series":"1.2.826.0.1.3680043.8.498.82238793131468220701507582136294353212","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489","series":"1.2.826.0.1.3680043.8.498.10587391702302335750652411177813729076","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38095098835657518803781210111463995296","series":"1.2.826.0.1.3680043.8.498.22420865979705111100354058550166822292","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290","series":"1.2.826.0.1.3680043.8.498.11364161624533341042650586367298779714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38238963443939714344730201261512911003","series":"1.2.826.0.1.3680043.8.498.13286557386035180461638347707293646845","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389","series":"1.2.826.0.1.3680043.8.498.32323997237906771515719220425087292546","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979","series":"1.2.826.0.1.3680043.8.498.86687780342070231920732170347655604522","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","series":"1.2.826.0.1.3680043.8.498.69915263316256545126962163675379858146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.47932241959578271007201940337201222157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082","series":"1.2.826.0.1.3680043.8.498.12091630513889093334890759673101925454","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38518069950656326194016175319535174299","series":"1.2.826.0.1.3680043.8.498.32290101526726317970857824609985834511","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267","series":"1.2.826.0.1.3680043.8.498.11792588086631092698865492181810701634","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715","series":"1.2.826.0.1.3680043.8.498.70068550396991489145269382748433924031","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451","series":"1.2.826.0.1.3680043.8.498.40841517457347881294782611236782006983","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598","series":"1.2.826.0.1.3680043.8.498.63105765726900772833538899488860715659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","series":"1.2.826.0.1.3680043.8.498.11974416109385803822822557601784594291","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893","series":"1.2.826.0.1.3680043.8.498.36071554355184637610618624163323625714","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.39960454289580310407361733465088811603","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455","series":"1.2.826.0.1.3680043.8.498.12798362081189482047812435954331521486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","series":"1.2.826.0.1.3680043.8.498.37894792761589055124747983072236302963","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718","series":"1.2.826.0.1.3680043.8.498.83695300173258284713186880915914054142","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948","series":"1.2.826.0.1.3680043.8.498.23959225273877216399938116018550799825","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","series":"1.2.826.0.1.3680043.8.498.11362518516887870044791589456798796440","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","series":"1.2.826.0.1.3680043.8.498.13359585557275365519038075908081788620","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39135770176150551756142329948474103964","series":"1.2.826.0.1.3680043.8.498.12167459513186122903832010581435745329","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584","series":"1.2.826.0.1.3680043.8.498.53430647148751100189447125585266300881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183","series":"1.2.826.0.1.3680043.8.498.51359341937435171167910228036245217049","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.79604097480577899455257123157291489279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402","series":"1.2.826.0.1.3680043.8.498.75506571074042745145888475077518759658","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027","series":"1.2.826.0.1.3680043.8.498.17737269200215083051360399650048128182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.12378818132915080063771897261355142120","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995","series":"1.2.826.0.1.3680043.8.498.40769194238348949646598382064970740157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.18686826006678133510693973875485801045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39895705984068714346274689335576595443","series":"1.2.826.0.1.3680043.8.498.15632228200284220590585986379810438569","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092","series":"1.2.826.0.1.3680043.8.498.56673254919745160266350425974711039077","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377","series":"1.2.826.0.1.3680043.8.498.42698530186990847442895258755849493137","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540","series":"1.2.826.0.1.3680043.8.498.13248116070745293528945248087568622499","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40196271329411009311273016603705560443","series":"1.2.826.0.1.3680043.8.498.34544365936324386836630379387573036682","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693","series":"1.2.826.0.1.3680043.8.498.37065809594905777720883405810121866375","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40257126480600927758308319831518876543","series":"1.2.826.0.1.3680043.8.498.75128805410596342459326377197100567138","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","series":"1.2.826.0.1.3680043.8.498.83600440023475903328475554590610820744","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.13187353452718568819540507069482635419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40437305053062892226978469238653825745","series":"1.2.826.0.1.3680043.8.498.60333369283594287716755350460980458658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161","series":"1.2.826.0.1.3680043.8.498.12737364540991312803704539662414426460","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40547250250566982489293057895661861722","series":"1.2.826.0.1.3680043.8.498.44428695130986927613627449536654149419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655","series":"1.2.826.0.1.3680043.8.498.48428783675329418004412952160224586543","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.58865718343375516309320108798512146002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.70427611448426663131001084053493803493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40765829900885943055669918091033014654","series":"1.2.826.0.1.3680043.8.498.36080662845624872107489131255977057976","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40828786094337186697222449293183309391","series":"1.2.826.0.1.3680043.8.498.89100853659082461572050157275057757302","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754","series":"1.2.826.0.1.3680043.8.498.39746519324467527155918430488863334275","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116","series":"1.2.826.0.1.3680043.8.498.10140083188814656590021764608926196067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994","series":"1.2.826.0.1.3680043.8.498.85997223158787518282049647464495630301","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388","series":"1.2.826.0.1.3680043.8.498.86973580053278530037652336327643543521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41197678541534079475667407055098243437","series":"1.2.826.0.1.3680043.8.498.57320242318260832144971781159767774314","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.33551558281661382017275298982420356816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615","series":"1.2.826.0.1.3680043.8.498.21269540090863000685167990639961559702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640","series":"1.2.826.0.1.3680043.8.498.40177602036263520145678442006723143216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764","series":"1.2.826.0.1.3680043.8.498.15180333711225942483738112554271372940","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049","series":"1.2.826.0.1.3680043.8.498.11603029539347581365578224668056371647","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41605033773891953796694219242501252567","series":"1.2.826.0.1.3680043.8.498.37391578895940181801495865849904375989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554","series":"1.2.826.0.1.3680043.8.498.88894569373213304493563628118937692498","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496","series":"1.2.826.0.1.3680043.8.498.29359490034090057476790526485274870128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.43027496214862870196529688959485840362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.46009729729396517437336430395185721517","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","series":"1.2.826.0.1.3680043.8.498.68336524232920097944945456223526527986","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004","series":"1.2.826.0.1.3680043.8.498.67040608429876800875456043774800181874","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41856137734822068230231773222820851271","series":"1.2.826.0.1.3680043.8.498.12359902290955149146159159334846949832","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810","series":"1.2.826.0.1.3680043.8.498.12410682734190081021064588099807594384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808","series":"1.2.826.0.1.3680043.8.498.34788384510574589442506546064622801858","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42120645248710860870761681999665860884","series":"1.2.826.0.1.3680043.8.498.93412585178898649026604113393640658408","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42162585695624963017992042571207363427","series":"1.2.826.0.1.3680043.8.498.96818469453867190129889352118522229326","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192","series":"1.2.826.0.1.3680043.8.498.11824966238060122925224631175913825069","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275","series":"1.2.826.0.1.3680043.8.498.23149315702749003724936618638938540965","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936","series":"1.2.826.0.1.3680043.8.498.19973602209437644060799618080031297780","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876","series":"1.2.826.0.1.3680043.8.498.11146120919470998861440232300258166338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42587087495729026068405824960397052997","series":"1.2.826.0.1.3680043.8.498.46446965534119686728782896352697046525","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017","series":"1.2.826.0.1.3680043.8.498.32251701004024308295116121333715518301","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42768603719459834345001093623865167866","series":"1.2.826.0.1.3680043.8.498.12909177574736982709304621111959067607","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394","series":"1.2.826.0.1.3680043.8.498.13081839058158279857249578785207185703","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42847376403722341195083350288786052637","series":"1.2.826.0.1.3680043.8.498.12258421979845969942108848180030181293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381","series":"1.2.826.0.1.3680043.8.498.79351386391002748618484408405580998151","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42942844751304929338846069579289911684","series":"1.2.826.0.1.3680043.8.498.69930957197206099939351756012158318551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295","series":"1.2.826.0.1.3680043.8.498.55890907476705739521308623110400970240","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43065391907984821806263204343792008938","series":"1.2.826.0.1.3680043.8.498.30200356594288837310744347353256980739","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364","series":"1.2.826.0.1.3680043.8.498.67956238741099934431871785436747676880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417","series":"1.2.826.0.1.3680043.8.498.11197079243588543550094093131404326036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684","series":"1.2.826.0.1.3680043.8.498.10995958839301553772542009858072196524","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43355923025859051425546451466843844346","series":"1.2.826.0.1.3680043.8.498.32995559924096839313521201030777389768","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684","series":"1.2.826.0.1.3680043.8.498.59591390931279217330821801849389852406","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490","series":"1.2.826.0.1.3680043.8.498.64411987083423429141679849854673258957","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43587826365647713044374316837330397375","series":"1.2.826.0.1.3680043.8.498.56376013327403787443556568457675358387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","series":"1.2.826.0.1.3680043.8.498.12964220574343488935331975601889903635","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43832771919542075229536000564653592465","series":"1.2.826.0.1.3680043.8.498.54321312892953197712163934313934985297","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44002376543340544410495269060824194475","series":"1.2.826.0.1.3680043.8.498.82036055747194726743961569774922615355","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587","series":"1.2.826.0.1.3680043.8.498.65197577514792655122359914581292176650","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44261256084247541720465769177956875383","series":"1.2.826.0.1.3680043.8.498.58737784464757879856745284357486670083","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455","series":"1.2.826.0.1.3680043.8.498.10111652225840538093245637319275535212","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474","series":"1.2.826.0.1.3680043.8.498.72877473025852424650319586862558977606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.11569123892617964372628169987514613652","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569","series":"1.2.826.0.1.3680043.8.498.46921258124003061210765894523160448798","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.12634449308435120288192845638483377482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530","series":"1.2.826.0.1.3680043.8.498.79122901428924593811756076046914302730","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803","series":"1.2.826.0.1.3680043.8.498.22460381378101812016089106286662739750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.12443035089620159632412763653165197462","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811","series":"1.2.826.0.1.3680043.8.498.37806336863958447452268470459028978044","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.55679553696082108073087926877155693871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083","series":"1.2.826.0.1.3680043.8.498.76871716109312510983507010176124522059","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634","series":"1.2.826.0.1.3680043.8.498.66178038399904756844301996088431483169","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293","series":"1.2.826.0.1.3680043.8.498.68509673080193666342035143580977107775","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45286493874121093392171044034263851961","series":"1.2.826.0.1.3680043.8.498.11692270206289371201120200995196779109","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492","series":"1.2.826.0.1.3680043.8.498.30195127915469798541627162498353337453","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","series":"1.2.826.0.1.3680043.8.498.60508153328877400477086621469417354872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616","series":"1.2.826.0.1.3680043.8.498.56668210276652084147572204176450619403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144","series":"1.2.826.0.1.3680043.8.498.80920639719359299385053370415671860747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552","series":"1.2.826.0.1.3680043.8.498.84329407701696665915923775292868748221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45519502487291848581539950159353124045","series":"1.2.826.0.1.3680043.8.498.13261071202956925361214714126622485414","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45644703214235636656820126625301033696","series":"1.2.826.0.1.3680043.8.498.78047710352926965521487426585476835449","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45755198489211010290512228906807170657","series":"1.2.826.0.1.3680043.8.498.11946113583198249453057934847940884914","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426","series":"1.2.826.0.1.3680043.8.498.11160048916956478244401384574176827167","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","series":"1.2.826.0.1.3680043.8.498.16342114870882716713253823912683259184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663","series":"1.2.826.0.1.3680043.8.498.51177427079477146083240681914337696057","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397","series":"1.2.826.0.1.3680043.8.498.10842149065587004589520919233498044670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995","series":"1.2.826.0.1.3680043.8.498.38582168257874606404890019649989598700","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856","series":"1.2.826.0.1.3680043.8.498.37884661432442056928333825613294777352","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46268820642949526111807948525363811264","series":"1.2.826.0.1.3680043.8.498.33516325023884607682531787533461516325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","series":"1.2.826.0.1.3680043.8.498.10369844332965475649555077068941146428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46335849264221717416382765091672040855","series":"1.2.826.0.1.3680043.8.498.75837938664125195260779715681446174654","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46452612704507986055186132736451572677","series":"1.2.826.0.1.3680043.8.498.78567345204921287992291661615364759313","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46506161284114284997220751358509505380","series":"1.2.826.0.1.3680043.8.498.13030955288510307853694311901187296457","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479","series":"1.2.826.0.1.3680043.8.498.69622581756209071404885264267059035198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241","series":"1.2.826.0.1.3680043.8.498.59648979593276993488419677734021663784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46797419946872257347118338518757027995","series":"1.2.826.0.1.3680043.8.498.23057658517714568961175469527632575373","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845","series":"1.2.826.0.1.3680043.8.498.21363141781435880627539647530128227980","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956","series":"1.2.826.0.1.3680043.8.498.85641871514214129985527264050034289269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.10758761211319401814577924762509425419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.83055380866142244102336620095225212158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972","series":"1.2.826.0.1.3680043.8.498.28698524388491641984346970183073400992","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004","series":"1.2.826.0.1.3680043.8.498.10023375006207708943543356180091297293","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47268599732083036147289040424392838620","series":"1.2.826.0.1.3680043.8.498.54334635823458915257239540040302725432","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.19280031721707746582650786478763752982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825","series":"1.2.826.0.1.3680043.8.498.17611824925456516260884288194004109089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246","series":"1.2.826.0.1.3680043.8.498.13116575153430483635389961146254595213","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400","series":"1.2.826.0.1.3680043.8.498.73002514475551845526732965991300384006","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","series":"1.2.826.0.1.3680043.8.498.13092950294203782360566037655689304231","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47855330524708374523226349383799712473","series":"1.2.826.0.1.3680043.8.498.91567473710585569295382775985895707391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187","series":"1.2.826.0.1.3680043.8.498.52680589140665925020263574237350014176","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835","series":"1.2.826.0.1.3680043.8.498.48261868502210520025020789077348738969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331","series":"1.2.826.0.1.3680043.8.498.26815173664619168044275719096395793248","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691","series":"1.2.826.0.1.3680043.8.498.10215576042285608398036280720762955520","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239","series":"1.2.826.0.1.3680043.8.498.12936188184179050131677321288106754777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","series":"1.2.826.0.1.3680043.8.498.44440297976353479716385458650094744906","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.17404727553361386264373847245843664551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48564780460694033783288540237012068889","series":"1.2.826.0.1.3680043.8.498.23298811603816021977134683725173541159","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236","series":"1.2.826.0.1.3680043.8.498.24924333016045895530220347377992267948","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560","series":"1.2.826.0.1.3680043.8.498.33205498022946058917568962416237535830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142","series":"1.2.826.0.1.3680043.8.498.27849574574696088329634362000326951169","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48872083192774100826667097552356058756","series":"1.2.826.0.1.3680043.8.498.96061274552772932267874280564645902165","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132","series":"1.2.826.0.1.3680043.8.498.25623620215370725395931151899931839246","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124","series":"1.2.826.0.1.3680043.8.498.34185171809287602928250484465343011393","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799","series":"1.2.826.0.1.3680043.8.498.55692087920350135862162000076461921359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49163335688297423808344168197551387860","series":"1.2.826.0.1.3680043.8.498.34081959981338237359337515926483899692","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295","series":"1.2.826.0.1.3680043.8.498.91673772831413896313422676521146821798","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638","series":"1.2.826.0.1.3680043.8.498.20170762064528969403940517026824374708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305","series":"1.2.826.0.1.3680043.8.498.13394738857053837072233267323762735842","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139","series":"1.2.826.0.1.3680043.8.498.66423208355433230860832897014234953799","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205","series":"1.2.826.0.1.3680043.8.498.73499102647279839850790976046083875600","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","series":"1.2.826.0.1.3680043.8.498.67623103161440943482632370016484869100","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.99396403950873899877341288566675056093","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921","series":"1.2.826.0.1.3680043.8.498.30041758532043603631248543482816802354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809","series":"1.2.826.0.1.3680043.8.498.72738405610181649228974106293331452143","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125","series":"1.2.826.0.1.3680043.8.498.16591562565805322340808157681642647528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021","series":"1.2.826.0.1.3680043.8.498.53385578468689069090891305974314156402","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49984662916977328110175445895515541267","series":"1.2.826.0.1.3680043.8.498.12835772386028361387691182536618836142","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.79421000550649637471681545010690132021","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50107479256735739327080544431929221135","series":"1.2.826.0.1.3680043.8.498.96933757089747970630958015983267917758","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333","series":"1.2.826.0.1.3680043.8.498.37013163087039628146716832751973724827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50215409013618353346619153290049205718","series":"1.2.826.0.1.3680043.8.498.12628176075712516429651985414499707676","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50259348432783983788186173025555513832","series":"1.2.826.0.1.3680043.8.498.12079663570307206507865308549202625210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50287512028986928372350279452197816661","series":"1.2.826.0.1.3680043.8.498.12725938952985371502446896240789468317","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321","series":"1.2.826.0.1.3680043.8.498.23364970215057399280655303691211042973","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.10665756076346231271025693656340246551","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.13065138465814951868935397318556890143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50648934951234229265309905009982297515","series":"1.2.826.0.1.3680043.8.498.96164563026534865186177204482850263014","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50667444467788034101157821576803778774","series":"1.2.826.0.1.3680043.8.498.76923615992056088532122013816421553561","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306","series":"1.2.826.0.1.3680043.8.498.27626938106222655670580463266746668516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241","series":"1.2.826.0.1.3680043.8.498.67829658131238489090847661534729877166","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","series":"1.2.826.0.1.3680043.8.498.55868343675708289388373775636506512126","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50819987120895374623148160166392914169","series":"1.2.826.0.1.3680043.8.498.10875003962414572338367928562011754308","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50844227724977857739867646104945090414","series":"1.2.826.0.1.3680043.8.498.51883147528639880045437886231399863980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50882628969985977810341532747520504841","series":"1.2.826.0.1.3680043.8.498.83448099025507486286056827036939096542","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324","series":"1.2.826.0.1.3680043.8.498.14605291189704223749101725969956020809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51054592064718900209123281525185013961","series":"1.2.826.0.1.3680043.8.498.11881723549017677657727056858549691889","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","series":"1.2.826.0.1.3680043.8.498.38711329343605176690484313173039639271","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.11762476813130838849251223954496892400","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995","series":"1.2.826.0.1.3680043.8.498.61078123116515799289436614587381565293","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51427159850769972151411336025891779423","series":"1.2.826.0.1.3680043.8.498.80511946408498641772072105913272124524","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967","series":"1.2.826.0.1.3680043.8.498.13152340972358762292429542374107210899","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.98889130881569759116298517963198446148","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131","series":"1.2.826.0.1.3680043.8.498.26155804725993873550289240215459438614","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51770131897828080663537020812721042704","series":"1.2.826.0.1.3680043.8.498.10297339405211309158790154832969666795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.18252978352352553012843974492886796309","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132","series":"1.2.826.0.1.3680043.8.498.76344723234116858490249692406387300513","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.14890193958380391952407071618599707993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965","series":"1.2.826.0.1.3680043.8.498.88455123424453359198910700228707638874","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51985313438970133264846259111827877590","series":"1.2.826.0.1.3680043.8.498.19869603817820367618688488337905208826","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761","series":"1.2.826.0.1.3680043.8.498.12457345522219685581768575524496394936","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52067134682915759489080529708019359378","series":"1.2.826.0.1.3680043.8.498.62361397809506111168170546557353614639","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52110163572817528199125320120779141456","series":"1.2.826.0.1.3680043.8.498.66568982282622558432296848309463745759","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.13149080221848076493519796331822282651","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055","series":"1.2.826.0.1.3680043.8.498.15212574381996762288122066818022884381","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","series":"1.2.826.0.1.3680043.8.498.61843133347052619406668335140042710104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52437948508356178175558637950043479411","series":"1.2.826.0.1.3680043.8.498.44479339403152795175857947511834833904","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661","series":"1.2.826.0.1.3680043.8.498.67534194468517418432836440681519811980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687","series":"1.2.826.0.1.3680043.8.498.10676182579683697932936705951369524200","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410","series":"1.2.826.0.1.3680043.8.498.31674819904390058482398058536421940639","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52785983814702054185130109942936184688","series":"1.2.826.0.1.3680043.8.498.12984061200219627731359156390367357745","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101","series":"1.2.826.0.1.3680043.8.498.33180731240994811271050349257993231330","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236","series":"1.2.826.0.1.3680043.8.498.65601863785369359262761197393693572845","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53076472372145933727990308381815269368","series":"1.2.826.0.1.3680043.8.498.82825633646933556987506092823126939770","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53080172184945962593713560671746700445","series":"1.2.826.0.1.3680043.8.498.11126004608448065578101652289332647606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847","series":"1.2.826.0.1.3680043.8.498.13491238298326362142040963736160920241","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782","series":"1.2.826.0.1.3680043.8.498.38909250269800336335703389189602053199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53221818460567216467597044606040911702","series":"1.2.826.0.1.3680043.8.498.81813722112094323552896312657574086495","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53271710112621372496324855809391580130","series":"1.2.826.0.1.3680043.8.498.10807407389534589430771334753698607480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262","series":"1.2.826.0.1.3680043.8.498.12201311195815242035806065399358139405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53361558447470435467999344560545063264","series":"1.2.826.0.1.3680043.8.498.47464682464388742634740123722096900479","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643","series":"1.2.826.0.1.3680043.8.498.60069200189321209894753992756486374568","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596","series":"1.2.826.0.1.3680043.8.498.11402231366625625884824685553273831056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423","series":"1.2.826.0.1.3680043.8.498.27211273684520777651232861760249295372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53737349894831365159124471910871800583","series":"1.2.826.0.1.3680043.8.498.99129326571628952254650798362504553371","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744","series":"1.2.826.0.1.3680043.8.498.12871811889674764755429496131879261176","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558","series":"1.2.826.0.1.3680043.8.498.64328063167434438509041913889700778143","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394","series":"1.2.826.0.1.3680043.8.498.30188671031722849793977935826683729508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.82366608131282271044299469149685051883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54100786607660749952814251186461344633","series":"1.2.826.0.1.3680043.8.498.76171897174781135076280689642714322768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54138747504377266740838999528964336002","series":"1.2.826.0.1.3680043.8.498.12469959542883328644346396113709575540","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54164004416963296439322252627774426375","series":"1.2.826.0.1.3680043.8.498.24032637303566047476168730525680429905","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043","series":"1.2.826.0.1.3680043.8.498.78009620482746158709864127970205226970","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232","series":"1.2.826.0.1.3680043.8.498.13009624693304210754183754064685178767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860","series":"1.2.826.0.1.3680043.8.498.69933519840760068086385269161536865704","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54448516923758211534546401490623824012","series":"1.2.826.0.1.3680043.8.498.45642726902848924654402350237526912337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299","series":"1.2.826.0.1.3680043.8.498.81817486283955667025289656878171977405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54589880111295324708950851057241927191","series":"1.2.826.0.1.3680043.8.498.12594676461936070219487345017496705431","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589","series":"1.2.826.0.1.3680043.8.498.77703395972488245430539939249235360602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.11738622147347322719722887551125671558","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446","series":"1.2.826.0.1.3680043.8.498.79598023624632399966282352802122356137","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469","series":"1.2.826.0.1.3680043.8.498.80529228095243449121410193639395379438","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757","series":"1.2.826.0.1.3680043.8.498.96361693368014829043404445733500274724","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883","series":"1.2.826.0.1.3680043.8.498.65338804688658931923600697615492278408","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.68390584919630341246382655832014279440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55067956175569778132589128971795053431","series":"1.2.826.0.1.3680043.8.498.53644928740766390325048060808891504977","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245","series":"1.2.826.0.1.3680043.8.498.55824424598012360477843192573932885628","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763","series":"1.2.826.0.1.3680043.8.498.78456007341293859282294166076225528867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55346033326903331672579723063380636237","series":"1.2.826.0.1.3680043.8.498.66890115949835740502212542554513199134","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55415479446253262335685494509470808516","series":"1.2.826.0.1.3680043.8.498.10939888829690263065288909185556342048","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","series":"1.2.826.0.1.3680043.8.498.51131149731184117257604679209077533059","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932","series":"1.2.826.0.1.3680043.8.498.11365543330464820848028216770245947881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.70290331533473067085813676251940404012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.86517327493008829773070508841390446019","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.32761503151252955560211310295181588965","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207","series":"1.2.826.0.1.3680043.8.498.32776836092155192850597157195372382334","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410","series":"1.2.826.0.1.3680043.8.498.12761331574901901295346433196293122797","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.21869235733835244812784241747370679125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.20301629736893509311089167842862415972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56091473614612257999053529249190958828","series":"1.2.826.0.1.3680043.8.498.13345971071496735484623019676151207995","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56156923369480999484275191793049573490","series":"1.2.826.0.1.3680043.8.498.67164751526366567364256619831235589565","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588","series":"1.2.826.0.1.3680043.8.498.93230953395834814248487356551249637061","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456","series":"1.2.826.0.1.3680043.8.498.36805601814571529155066045849824648370","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770","series":"1.2.826.0.1.3680043.8.498.45866396481374257394181209861775492170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56506415277488978301157490514745400410","series":"1.2.826.0.1.3680043.8.498.64704999575263864342371255879009837904","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.47733863137654577393750479312116091574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923","series":"1.2.826.0.1.3680043.8.498.64190750745759895410391437300332817006","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56715311294415711253166312546942356882","series":"1.2.826.0.1.3680043.8.498.11237417460819573480117289441316822766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070","series":"1.2.826.0.1.3680043.8.498.51310570190384949846363886186065076487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033","series":"1.2.826.0.1.3680043.8.498.85851063548292299849789033230618280275","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883","series":"1.2.826.0.1.3680043.8.498.98662226813349252741782702121736340145","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068","series":"1.2.826.0.1.3680043.8.498.41270106013747714203277039860925942398","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026","series":"1.2.826.0.1.3680043.8.498.17765505690924022616453303633583234022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","series":"1.2.826.0.1.3680043.8.498.12106198301877411229768365241561535563","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56993108275828399240508695615212151102","series":"1.2.826.0.1.3680043.8.498.24981336315328645089927905067337895557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369","series":"1.2.826.0.1.3680043.8.498.35062300206106434732040282427132577563","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856","series":"1.2.826.0.1.3680043.8.498.10732592498520187255103194080654465230","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57316522861709598765962486949846744486","series":"1.2.826.0.1.3680043.8.498.47654458477334406679634457003805158119","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","series":"1.2.826.0.1.3680043.8.498.23311409471739444748914996719370075235","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57537915810250993237120953960404180449","series":"1.2.826.0.1.3680043.8.498.68511822910280437960059453589495153133","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454","series":"1.2.826.0.1.3680043.8.498.17884987448122685564081849213202580825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.10482953523378544223931005096908522838","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57811199787891897795884073594875380873","series":"1.2.826.0.1.3680043.8.498.10789368838532827352087886587417883995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57845032486422442404222046144765204905","series":"1.2.826.0.1.3680043.8.498.40194777312534987382525856421558280196","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977","series":"1.2.826.0.1.3680043.8.498.13393645696795033569922380286393570826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.48716803025076849349075102564185349837","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887","series":"1.2.826.0.1.3680043.8.498.60694424501282146497298781452156482215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58261214953729037661916565473101636690","series":"1.2.826.0.1.3680043.8.498.33583406378919297597672009564266929008","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195","series":"1.2.826.0.1.3680043.8.498.10711725310826017239803054281481097757","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58366682083795592678613986311570432419","series":"1.2.826.0.1.3680043.8.498.61808750231374203078296662927778710916","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292","series":"1.2.826.0.1.3680043.8.498.48292744802004562594687184388921784199","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621","series":"1.2.826.0.1.3680043.8.498.31374039250188591515475419661288571349","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644","series":"1.2.826.0.1.3680043.8.498.20111252001646255767647281146178472655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434","series":"1.2.826.0.1.3680043.8.498.11331025145329479934244925872126820705","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101","series":"1.2.826.0.1.3680043.8.498.29760893566643123692846645487414816426","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","series":"1.2.826.0.1.3680043.8.498.10132705216136327122647064228576425214","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744","series":"1.2.826.0.1.3680043.8.498.61416922003109465449982363520392027035","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130","series":"1.2.826.0.1.3680043.8.498.11110555413446607488670457580668434485","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538","series":"1.2.826.0.1.3680043.8.498.61230263310011490307854581684787496955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.60501414438703611066075130873777295350","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.12441029408199554566099888158052974975","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.12131775322401053354204851273685102725","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.28109800190578422428891621550315961261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612","series":"1.2.826.0.1.3680043.8.498.69345645595995319357318252712467413020","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59309239681970150083311665704264518844","series":"1.2.826.0.1.3680043.8.498.11093445231101139020606240316636405681","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823","series":"1.2.826.0.1.3680043.8.498.26459848109229247943617962612989887082","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59475691595504590121311599301170361869","series":"1.2.826.0.1.3680043.8.498.10002261797060075169053014192535848502","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710","series":"1.2.826.0.1.3680043.8.498.13025502148523091259027913261486376465","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.78355166731578366716311958391884937298","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.73944641199331487153960010681768736672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59826277112503440743678968423276737254","series":"1.2.826.0.1.3680043.8.498.95203696267217101578819133649754209412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439","series":"1.2.826.0.1.3680043.8.498.76879808207172545114053432563832900218","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.54039885251747185965995391144531491053","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226","series":"1.2.826.0.1.3680043.8.498.10883585244506773525002187708928084863","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.10062461076000933362424628266183945811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357","series":"1.2.826.0.1.3680043.8.498.12564338962380660104398914501736510117","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162","series":"1.2.826.0.1.3680043.8.498.84864405435435772039576049660499261823","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.76676439273089505830842674536172110808","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044","series":"1.2.826.0.1.3680043.8.498.12247005486278323975267197657529907423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60430475172382878092821699001847797153","series":"1.2.826.0.1.3680043.8.498.95218893279040590699807385689176575009","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","series":"1.2.826.0.1.3680043.8.498.11696399086029771950703474006966450398","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190","series":"1.2.826.0.1.3680043.8.498.12714310266665605619269022694808522530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109","series":"1.2.826.0.1.3680043.8.498.78274209383277324011488394624406056052","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384","series":"1.2.826.0.1.3680043.8.498.12283731182825317973756548021828736514","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589","series":"1.2.826.0.1.3680043.8.498.28555365105904559548192843807392628434","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60723378470101669902158229125336455520","series":"1.2.826.0.1.3680043.8.498.11690150302853011274812674183746484745","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592","series":"1.2.826.0.1.3680043.8.498.10878468862661041952413841181685212894","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873","series":"1.2.826.0.1.3680043.8.498.69670870235549888607634451612530780214","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.45011939244154780930592418743327137113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106","series":"1.2.826.0.1.3680043.8.498.46531510740303929002967214503873105027","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841","series":"1.2.826.0.1.3680043.8.498.10801104133062503440368565680255667795","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675","series":"1.2.826.0.1.3680043.8.498.33448934914994928241887305047542101589","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563","series":"1.2.826.0.1.3680043.8.498.32420913085631547269285815155747635868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61456481310419978302697858056095086811","series":"1.2.826.0.1.3680043.8.498.65388529161055471473738685933887350825","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.10824313044060519376945112910546901326","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.32377952812776135942021080232010442744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61630894340791204767579628554826500984","series":"1.2.826.0.1.3680043.8.498.12085209220414462776200150219589083330","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61797111379749329977228618245440305623","series":"1.2.826.0.1.3680043.8.498.36818354655502267882839602860088899290","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61904856234837992279531381374350177464","series":"1.2.826.0.1.3680043.8.498.89368335278210270837249119024214933570","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61981891095235024266625598999854298547","series":"1.2.826.0.1.3680043.8.498.13191292268452590750511010391078469432","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008","series":"1.2.826.0.1.3680043.8.498.34596199416494017047882974291988170460","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.64365287922653646025790845391526317477","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62217447760619326492634156203608889396","series":"1.2.826.0.1.3680043.8.498.61630146830328264673714236729468369777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.32446494160670291371172728822249794058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097","series":"1.2.826.0.1.3680043.8.498.10414198041768076563354889874801298522","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62400625562782184774973349983498962743","series":"1.2.826.0.1.3680043.8.498.60740417675349165837376054103159318081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","series":"1.2.826.0.1.3680043.8.498.48311102850450655830386776465478701880","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.81892732695419468989322862582884021558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62752755711205966800746994120425231855","series":"1.2.826.0.1.3680043.8.498.89332966940075361205906558924594902848","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504","series":"1.2.826.0.1.3680043.8.498.96143722249441523004920054532195972592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916","series":"1.2.826.0.1.3680043.8.498.11146853011455769002225072436737910628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.58841061018874545455578167670723747217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682","series":"1.2.826.0.1.3680043.8.498.11342530492369540120678844067297890906","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.58438023206344872937339343511656311444","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410","series":"1.2.826.0.1.3680043.8.498.92865247620345064195957269277932570621","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997","series":"1.2.826.0.1.3680043.8.498.13390502300674904842844497454378311860","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181","series":"1.2.826.0.1.3680043.8.498.11644178139328205862041741547784465640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.66255294615516352876359775439913821321","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540","series":"1.2.826.0.1.3680043.8.498.79777972332314840267624650987994912722","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.31317792374706468755947510841492326936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63424633611847022112809173449690688015","series":"1.2.826.0.1.3680043.8.498.75225873115674512088587486614118614412","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63439188800533762543950349022148547855","series":"1.2.826.0.1.3680043.8.498.30219529160453052561435380589202631185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343","series":"1.2.826.0.1.3680043.8.498.20662510378113599088419165842419465935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480","series":"1.2.826.0.1.3680043.8.498.69563895284921360753339319081415669566","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731","series":"1.2.826.0.1.3680043.8.498.25465153063370534263257208724601281353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.87569874613469191329718338899389367069","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809","series":"1.2.826.0.1.3680043.8.498.72670097160866711668146661490063373489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652","series":"1.2.826.0.1.3680043.8.498.41174520385515855235458840416465973086","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566","series":"1.2.826.0.1.3680043.8.498.61443777511115808406051530682199077670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64022949342538714157641824612944253302","series":"1.2.826.0.1.3680043.8.498.90055663962881372374705678351299936229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567","series":"1.2.826.0.1.3680043.8.498.58828232864393461135515052189720229850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.10192777225050747611729749702310348730","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","series":"1.2.826.0.1.3680043.8.498.10996717405139544747172286881682362734","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715","series":"1.2.826.0.1.3680043.8.498.55958625052051868575436515541267093833","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","series":"1.2.826.0.1.3680043.8.498.78235179568036527421818159538436473078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676","series":"1.2.826.0.1.3680043.8.498.54830105120608560737425666759105156780","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.13357026917304822103046333628335055127","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64527059976573114931020581328924993069","series":"1.2.826.0.1.3680043.8.498.11943684249447864864435530541386810961","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.64606794953061143951972249525094490665","series":"1.2.826.0.1.3680043.8.498.69803400596076860186003754059486495273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188","series":"1.2.826.0.1.3680043.8.498.36732567656181545758023489459848729705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048","series":"1.2.826.0.1.3680043.8.498.34235424404099712614359494985316999534","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64897285598346101578442697515294908341","series":"1.2.826.0.1.3680043.8.498.90343010537349440631397592642611423117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183","series":"1.2.826.0.1.3680043.8.498.10829518572080462116197141483348107931","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65085954283273471512936799274064949557","series":"1.2.826.0.1.3680043.8.498.96259744002827077874749512532297853303","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405","series":"1.2.826.0.1.3680043.8.498.79373480546531168752850861958351215835","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","series":"1.2.826.0.1.3680043.8.498.10687946708881947473023379051556772839","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549","series":"1.2.826.0.1.3680043.8.498.38497707940468039422233351382057604976","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.55326519435926380235406538380353327407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554","series":"1.2.826.0.1.3680043.8.498.96098126818366673434774108286554299952","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","series":"1.2.826.0.1.3680043.8.498.36480041806010774525985248893473089744","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65547889329285432771702699287910809921","series":"1.2.826.0.1.3680043.8.498.12276218609335918444947846268031018813","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65667678378249093118816281743956646856","series":"1.2.826.0.1.3680043.8.498.41333695866095814903424109393898891774","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327","series":"1.2.826.0.1.3680043.8.498.12032322953692219076472026507143127816","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65763261177572072944828171262268039852","series":"1.2.826.0.1.3680043.8.498.49073614000792287148734700515876257081","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505","series":"1.2.826.0.1.3680043.8.498.12027151130034168067578359372062991521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786","series":"1.2.826.0.1.3680043.8.498.77970238174499068090093978314784081937","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65896962360688978581417748865609785541","series":"1.2.826.0.1.3680043.8.498.22238295948969349151000267480091533172","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593","series":"1.2.826.0.1.3680043.8.498.13131974380712473195850967966021132563","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921","series":"1.2.826.0.1.3680043.8.498.45177861295103560496373520447343650298","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66065279867975557954375204776779208827","series":"1.2.826.0.1.3680043.8.498.42678124440573597254842073131429585165","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","series":"1.2.826.0.1.3680043.8.498.10545270718535988613361899072404971224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769","series":"1.2.826.0.1.3680043.8.498.43296195174962803172031900185320419606","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084","series":"1.2.826.0.1.3680043.8.498.11384086304522261622818486120566121463","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66308195780676793917989368423009849637","series":"1.2.826.0.1.3680043.8.498.58012509431712373414158226929588185032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695","series":"1.2.826.0.1.3680043.8.498.12383249811069358589273962576781399085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421","series":"1.2.826.0.1.3680043.8.498.23934541564833332807502725182368225009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66510565421813721258189101326477318907","series":"1.2.826.0.1.3680043.8.498.12852280776066477604639737460239998605","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726","series":"1.2.826.0.1.3680043.8.498.13252695847139007811285851447755259055","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853","series":"1.2.826.0.1.3680043.8.498.86383461942619363712945462397217100021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084","series":"1.2.826.0.1.3680043.8.498.76011673612852617382227676263562684435","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647","series":"1.2.826.0.1.3680043.8.498.18668357472811035272629456910023090899","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710","series":"1.2.826.0.1.3680043.8.498.11440589521369316786417618218835729125","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924","series":"1.2.826.0.1.3680043.8.498.73994163153160474750415445103506974696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967","series":"1.2.826.0.1.3680043.8.498.19579272912695787649569287071114559363","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258","series":"1.2.826.0.1.3680043.8.498.17655080056113700700710333302799713806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123","series":"1.2.826.0.1.3680043.8.498.98457614909176907401088310140777556622","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784","series":"1.2.826.0.1.3680043.8.498.11216751253159669787065217045241323169","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.12303640761135542651539013526370733030","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67527486841399075490744039058207332725","series":"1.2.826.0.1.3680043.8.498.34076076230212720000740204602984872494","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67601368724828563564325678719126190797","series":"1.2.826.0.1.3680043.8.498.43950937581691733597041304647798853963","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451","series":"1.2.826.0.1.3680043.8.498.81817014253824809099659255117322699085","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172","series":"1.2.826.0.1.3680043.8.498.88608487612293430040290915018554598707","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814","series":"1.2.826.0.1.3680043.8.498.11451933712824617963972109165694487139","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378","series":"1.2.826.0.1.3680043.8.498.25310710995446834610100352042814834638","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67817378785531423675449308236680249899","series":"1.2.826.0.1.3680043.8.498.95414086852935200731373546880527851523","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67901792488494108577057563458320460313","series":"1.2.826.0.1.3680043.8.498.12305889296427963457511115252977542047","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.70917587410073221467214440250170042637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.27566092550582715220100986187512089331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68042422993686716523227090971667037309","series":"1.2.826.0.1.3680043.8.498.26811453926210319576114015874060952285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771","series":"1.2.826.0.1.3680043.8.498.23634252064212322310775716267219415559","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","series":"1.2.826.0.1.3680043.8.498.12561323352343097281936458019933968243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.10370798628018548953580678824040812459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68176786698913278568384612123261229708","series":"1.2.826.0.1.3680043.8.498.95295995762689824698933408968591862405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68236154531625015457518732674625182872","series":"1.2.826.0.1.3680043.8.498.91793146126001052208773369331966581131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645","series":"1.2.826.0.1.3680043.8.498.35450028079776013116651987917484251169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359","series":"1.2.826.0.1.3680043.8.498.84856935910543376412603608882157552351","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68434323590918408241182252996136001891","series":"1.2.826.0.1.3680043.8.498.27458924203645694591026627255994394021","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68442883836038453562548037358230921748","series":"1.2.826.0.1.3680043.8.498.33202698033821240712450933766713584573","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.31725279893720226999009822804215429007","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028","series":"1.2.826.0.1.3680043.8.498.40716597214956572193006844695212941528","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643","series":"1.2.826.0.1.3680043.8.498.43379025446875214272088597443050962675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636","series":"1.2.826.0.1.3680043.8.498.10150968935718101303542096896860079333","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522","series":"1.2.826.0.1.3680043.8.498.35475698516127388191488706984816169817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68799355386870083063665227055340582612","series":"1.2.826.0.1.3680043.8.498.58303791169596889439171852079896802745","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.10203353291428924408099807238468662354","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471","series":"1.2.826.0.1.3680043.8.498.11318398718544701982892083112734200242","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840","series":"1.2.826.0.1.3680043.8.498.69648607576146715275663832571979455767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69131433601281694165384475765166382765","series":"1.2.826.0.1.3680043.8.498.12625812197035178886041783662905400263","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.44357996484223408424632186229569019769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69260280362382086729793018979234418276","series":"1.2.826.0.1.3680043.8.498.95473668790286967865934329629547147383","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002","series":"1.2.826.0.1.3680043.8.498.16539998982255496119019092775291142745","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359","series":"1.2.826.0.1.3680043.8.498.12629423041849186473119818542688479555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69470130842235635628917024269527211321","series":"1.2.826.0.1.3680043.8.498.12806569437554616943325611270519063177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339","series":"1.2.826.0.1.3680043.8.498.42106980538344026966021312823386396825","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69632034454628807663252084651278084839","series":"1.2.826.0.1.3680043.8.498.10435077214403290782535184882625619919","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69665129458102359604411189244405850661","series":"1.2.826.0.1.3680043.8.498.87655155486239550585489708892671368627","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813","series":"1.2.826.0.1.3680043.8.498.65045849114806956013149606324332694233","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69755849784902271124335765927125722483","series":"1.2.826.0.1.3680043.8.498.10028537904773381725019327429368094016","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69917156795941398965793962392683899162","series":"1.2.826.0.1.3680043.8.498.15617451660086788259068894201486428319","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69954533310505596128507234525019777805","series":"1.2.826.0.1.3680043.8.498.15407752643441383182744250201877231600","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70129413051088864999976563404818211137","series":"1.2.826.0.1.3680043.8.498.12246908797370365702062481912839091134","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70271042033022764422915852922035168712","series":"1.2.826.0.1.3680043.8.498.36535322152151139065098950399754804294","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063","series":"1.2.826.0.1.3680043.8.498.19658365047044220895317338395663891174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70375834273924102402706635540898552043","series":"1.2.826.0.1.3680043.8.498.55591315804043779827638854272996354654","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70435722790459313233345196619727661231","series":"1.2.826.0.1.3680043.8.498.68339479344564394993933502049673133772","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757","series":"1.2.826.0.1.3680043.8.498.11251781324733769945191635991162520843","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70531311408047627903498280968772932092","series":"1.2.826.0.1.3680043.8.498.78603731580122685084310432828217191225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395","series":"1.2.826.0.1.3680043.8.498.31180848926036063191639773343991579822","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717","series":"1.2.826.0.1.3680043.8.498.12597025200078329982195724481313100184","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70829434408808897601630953382239533220","series":"1.2.826.0.1.3680043.8.498.21844599693189787794498844467749441496","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541","series":"1.2.826.0.1.3680043.8.498.80832994079284592208092526120319740952","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.10416648697509247566721762644203814620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068","series":"1.2.826.0.1.3680043.8.498.10730139647774228020949102379370669265","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","series":"1.2.826.0.1.3680043.8.498.21047964874169368078957202496975868526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316","series":"1.2.826.0.1.3680043.8.498.90976049583701982222747711366528302686","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71001339006150287453095055802545712130","series":"1.2.826.0.1.3680043.8.498.97304422562269897581968210898759028240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71084400653875583977681616665093993652","series":"1.2.826.0.1.3680043.8.498.77160452879762089681882818389497350455","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999","series":"1.2.826.0.1.3680043.8.498.50062699084829342820856592482266447942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215","series":"1.2.826.0.1.3680043.8.498.20602786464236580808837031919678280234","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.11198006896255214253225008693981398381","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629","series":"1.2.826.0.1.3680043.8.498.91009726175606105184928439767415944294","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71671013221669628598634284971059228267","series":"1.2.826.0.1.3680043.8.498.81853051453571777751048765075392339410","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71688651674041466985305836912343513997","series":"1.2.826.0.1.3680043.8.498.12720228687087771317756972125582026592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656","series":"1.2.826.0.1.3680043.8.498.11897659536098001365015399474047499326","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71744523263023933066263506201344989825","series":"1.2.826.0.1.3680043.8.498.13391311441910057610345794057710664717","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71766161038805688022754893134126528749","series":"1.2.826.0.1.3680043.8.498.54142391633575244084662223909375403384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71814919885777053270239861047114794237","series":"1.2.826.0.1.3680043.8.498.11839463441005059373773553289357871170","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.52730766500471877220980063050997976767","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71899370657875960028369229847153071577","series":"1.2.826.0.1.3680043.8.498.32580068542548579218956124639884988739","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72008869367139958325145888733651022033","series":"1.2.826.0.1.3680043.8.498.73785580098879090527635695101860249619","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72018602547183896629903902899903468287","series":"1.2.826.0.1.3680043.8.498.64063980833592753642513286236139126667","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529","series":"1.2.826.0.1.3680043.8.498.40974010997607966049465799972463032132","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641","series":"1.2.826.0.1.3680043.8.498.76240945693310477871976969778147465919","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72237065473467452226073540784070938342","series":"1.2.826.0.1.3680043.8.498.58832602953272875855120068507193424194","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753","series":"1.2.826.0.1.3680043.8.498.94383690566585988918850204082334654757","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687","series":"1.2.826.0.1.3680043.8.498.46808689039463165076787209115781574008","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572","series":"1.2.826.0.1.3680043.8.498.84762791482554265931423159030346973956","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.12482877642970985094139686846833486447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72494404099796908482534991590660770670","series":"1.2.826.0.1.3680043.8.498.64372228521266653941259639941815202071","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72495677804641720406765749568114476096","series":"1.2.826.0.1.3680043.8.498.45917910067635865066576195454329047258","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534","series":"1.2.826.0.1.3680043.8.498.11578804148297149694999195892827777788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540","series":"1.2.826.0.1.3680043.8.498.64633179199518212558497804493312217849","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585","series":"1.2.826.0.1.3680043.8.498.36334047859558987107972714108384989189","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72786773761211751423503785260617678955","series":"1.2.826.0.1.3680043.8.498.74967554575366267685246446631770018326","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230","series":"1.2.826.0.1.3680043.8.498.13105606039283435988698078089239464260","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813","series":"1.2.826.0.1.3680043.8.498.13910863466102002293956688217478377953","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492","series":"1.2.826.0.1.3680043.8.498.75406192340009918044095407934925282827","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331","series":"1.2.826.0.1.3680043.8.498.67920434964954730640863179549043521793","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408","series":"1.2.826.0.1.3680043.8.498.12566228508645398260356755233154671260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.94614389182187140376995937982853833425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642","series":"1.2.826.0.1.3680043.8.498.92895738283788419878732456608523326779","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266","series":"1.2.826.0.1.3680043.8.498.12288900994494330717441968693335519562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894","series":"1.2.826.0.1.3680043.8.498.31804726826188087837159062206656390533","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158","series":"1.2.826.0.1.3680043.8.498.80476316067957261550828909209531854321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930","series":"1.2.826.0.1.3680043.8.498.31054636126570881375178752209438074369","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.84475987492464622019588339023784873363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73394992092536626285935384324191560182","series":"1.2.826.0.1.3680043.8.498.77013812090358928466748563853644130704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803","series":"1.2.826.0.1.3680043.8.498.64446664756608201015020196825443251579","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.99706895063734370111952508133877314975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951","series":"1.2.826.0.1.3680043.8.498.51283249486677736728909435938079486478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605","series":"1.2.826.0.1.3680043.8.498.11232152515030548730978852729127839380","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73716752081143767997507440278231972668","series":"1.2.826.0.1.3680043.8.498.59569406427206566797539195688592623431","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73792361507751648386401969571717557389","series":"1.2.826.0.1.3680043.8.498.36289209514637118116322185869367279418","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063","series":"1.2.826.0.1.3680043.8.498.12146698071911096779206485155138043336","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980","series":"1.2.826.0.1.3680043.8.498.60036837134643607081892319547266165166","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74078340346189210015785395961373387782","series":"1.2.826.0.1.3680043.8.498.11778759923828054380073115882756141928","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520","series":"1.2.826.0.1.3680043.8.498.91578916219550543934165640458194707615","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558","series":"1.2.826.0.1.3680043.8.498.73731190660949263264746606312141548539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186","series":"1.2.826.0.1.3680043.8.498.88777860621520800354967737588212366711","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604","series":"1.2.826.0.1.3680043.8.498.98388037524820894573995410735703570809","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74243564762473331581882860440560337725","series":"1.2.826.0.1.3680043.8.498.24734937345610572475613116198088518182","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","series":"1.2.826.0.1.3680043.8.498.10235178261166447072383464548264334309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860","series":"1.2.826.0.1.3680043.8.498.76995936838348964315256433350844648814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.87362573879657017242198301737188759523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471396469132211069109289423495141062","series":"1.2.826.0.1.3680043.8.498.72463289140677850716794537125028213677","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74622912009754642425258484521795653023","series":"1.2.826.0.1.3680043.8.498.17005134178773557812792527935871421404","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373","series":"1.2.826.0.1.3680043.8.498.87645552318874307353920489616595827087","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.46582862072353597965813760762140022350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866","series":"1.2.826.0.1.3680043.8.498.43686313922874748535298202050709379299","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74939269621045823332837094516056211606","series":"1.2.826.0.1.3680043.8.498.70411760304875226457309476354613240613","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75067227246352035607675978826177487284","series":"1.2.826.0.1.3680043.8.498.88276429824772293550786769118334466359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.92727942743325700702271100407157210902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309","series":"1.2.826.0.1.3680043.8.498.84609387082130830237494518509865804460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638","series":"1.2.826.0.1.3680043.8.498.19374493949415168209516540663169341568","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75418637314688036152637525301534342254","series":"1.2.826.0.1.3680043.8.498.79496084315235312081750968993999225254","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846","series":"1.2.826.0.1.3680043.8.498.81979765295160238740836450309982669066","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75522002047190283617659083787085821064","series":"1.2.826.0.1.3680043.8.498.11619821081502619032854393342526027530","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554","series":"1.2.826.0.1.3680043.8.498.71392481354162508262877564605966814692","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.12427724921301216345409085679536981444","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.39848840520587744141278111230939535107","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535","series":"1.2.826.0.1.3680043.8.498.10818930179772013200574780868160648582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75746520601616488682662195284941583868","series":"1.2.826.0.1.3680043.8.498.11183258145338081360823789433065308146","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447","series":"1.2.826.0.1.3680043.8.498.37613206790496647960241733886130432151","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877","series":"1.2.826.0.1.3680043.8.498.17496798231473199222326687086736663170","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75976388060969166980873144500539735332","series":"1.2.826.0.1.3680043.8.498.73469106580073495733210218468772748093","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76012896432318288623173593280895772876","series":"1.2.826.0.1.3680043.8.498.88781616426285659497473581232750901927","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523","series":"1.2.826.0.1.3680043.8.498.11422056993694193666321666760553223762","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.98291179156152245398289998234662621165","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625","series":"1.2.826.0.1.3680043.8.498.68731955626749966616069499624871425319","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76242245340157474968574926079920787199","series":"1.2.826.0.1.3680043.8.498.34140873614326231265540908649901212717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379","series":"1.2.826.0.1.3680043.8.498.11496388328656249236457113897484779959","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024","series":"1.2.826.0.1.3680043.8.498.99229387710488587230665742079671620140","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992","series":"1.2.826.0.1.3680043.8.498.20433683361118885816441472691189006984","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636","series":"1.2.826.0.1.3680043.8.498.12924792612654051520057028064144235818","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76742750470853901575934097885382242952","series":"1.2.826.0.1.3680043.8.498.40047325088166240256573293840282049554","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","series":"1.2.826.0.1.3680043.8.498.72665234300357370378327513257915742353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737","series":"1.2.826.0.1.3680043.8.498.72967047023078269093814687375821244488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939","series":"1.2.826.0.1.3680043.8.498.12083509548115293078424637372319468202","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223","series":"1.2.826.0.1.3680043.8.498.83463350474245594019012621694804973714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77058587552329832558354544874120592826","series":"1.2.826.0.1.3680043.8.498.12988845761553273160154681925560074014","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536","series":"1.2.826.0.1.3680043.8.498.11233515653100249977861545953174784117","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733","series":"1.2.826.0.1.3680043.8.498.13050022020759905604085753048928038464","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.88843957487586326950967944854999975727","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","series":"1.2.826.0.1.3680043.8.498.10617683740053045766734445454196809519","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300","series":"1.2.826.0.1.3680043.8.498.56530539928346131092148404389550664874","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485","series":"1.2.826.0.1.3680043.8.498.29474873665144013324278411612795818180","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.99177866403443575396896317080757605662","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012","series":"1.2.826.0.1.3680043.8.498.65290870189985700398419808420792140456","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512","series":"1.2.826.0.1.3680043.8.498.71805342511900542452370329506031278555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.68197507127679407142312412117547159147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587","series":"1.2.826.0.1.3680043.8.498.11607187723916932442712204610212319043","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77524507552350448792970399203131998592","series":"1.2.826.0.1.3680043.8.498.41914118992229293512981556222390506883","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713","series":"1.2.826.0.1.3680043.8.498.15576758637798824773691030673453186535","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777","series":"1.2.826.0.1.3680043.8.498.12286395481155776691859658904164533047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176","series":"1.2.826.0.1.3680043.8.498.51185248633051937755011684122514132886","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488","series":"1.2.826.0.1.3680043.8.498.13331691187814285255553485665880891202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873","series":"1.2.826.0.1.3680043.8.498.12983118095869679272698603459738174559","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283","series":"1.2.826.0.1.3680043.8.498.66400113188286749771125710601208241672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497","series":"1.2.826.0.1.3680043.8.498.47016712959698847916625888375161097784","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026","series":"1.2.826.0.1.3680043.8.498.32755148224465731128689066523902300477","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.15041177639595349578424046121434219552","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429","series":"1.2.826.0.1.3680043.8.498.11672830721872529964042652453194960814","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437","series":"1.2.826.0.1.3680043.8.498.10167499509779125238005757260535903279","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411","series":"1.2.826.0.1.3680043.8.498.37874691952016285472892559731121125593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293","series":"1.2.826.0.1.3680043.8.498.74491083343510839514910199530655617356","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464","series":"1.2.826.0.1.3680043.8.498.10607641229552540485430068795562255368","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78420763928408680447797586163977368190","series":"1.2.826.0.1.3680043.8.498.11869781062324807115177726853345481090","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568","series":"1.2.826.0.1.3680043.8.498.28374105314719139335552344448519556342","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196","series":"1.2.826.0.1.3680043.8.498.53351551074089569967518897008551700762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756","series":"1.2.826.0.1.3680043.8.498.12920305088284629571701801349698533991","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78674598286586525734293563394409166190","series":"1.2.826.0.1.3680043.8.498.10795078734885659291110323925205950669","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453","series":"1.2.826.0.1.3680043.8.498.41638077056982146688098193784487146357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555","series":"1.2.826.0.1.3680043.8.498.78176092794245958588048841255448253934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949","series":"1.2.826.0.1.3680043.8.498.31335462820753967379080259325640445834","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755","series":"1.2.826.0.1.3680043.8.498.40587048563462161528460859595699287917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611","series":"1.2.826.0.1.3680043.8.498.94355506852706355784981627712103069457","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986","series":"1.2.826.0.1.3680043.8.498.41264372771525617120086989530114799719","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.40159196130767835928052774663568235857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718","series":"1.2.826.0.1.3680043.8.498.29762565424597263620550482216584812874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348","series":"1.2.826.0.1.3680043.8.498.57579760498458776539426255241217978940","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192","series":"1.2.826.0.1.3680043.8.498.30526556458197364057368353610203229222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674","series":"1.2.826.0.1.3680043.8.498.40523480696153355215382031458358543947","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079","series":"1.2.826.0.1.3680043.8.498.15550343542175508722274500853946552177","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803","series":"1.2.826.0.1.3680043.8.498.80888594293756675639771681809241331389","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79582457162604674356422857305405255763","series":"1.2.826.0.1.3680043.8.498.47306027039454630242223140771880260351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426","series":"1.2.826.0.1.3680043.8.498.19858395664842839830665886668897167990","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","series":"1.2.826.0.1.3680043.8.498.12380431690809899560619025766259112214","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290","series":"1.2.826.0.1.3680043.8.498.48560868112891235060020380068279460415","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900","series":"1.2.826.0.1.3680043.8.498.44088926208064890052598079047281916658","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934","series":"1.2.826.0.1.3680043.8.498.24551556137334536885805170720131200678","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80180578697901833764761405542790185180","series":"1.2.826.0.1.3680043.8.498.43565631876717056152253714924501430166","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.37026520093783663119780058345318987446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798","series":"1.2.826.0.1.3680043.8.498.57325438173567431191970008349021889974","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111","series":"1.2.826.0.1.3680043.8.498.12127379290950935483209115727842056867","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146","series":"1.2.826.0.1.3680043.8.498.12015617538085489240815801351548358804","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80433141990021472112645841723991584182","series":"1.2.826.0.1.3680043.8.498.12337469633526968887927228573340976075","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561","series":"1.2.826.0.1.3680043.8.498.79788923044623920357636652713001220742","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003","series":"1.2.826.0.1.3680043.8.498.68053739327458784956673123202758130300","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647","series":"1.2.826.0.1.3680043.8.498.85733529822693643868048384338888332078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.10297620388648925033232776540994538244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506","series":"1.2.826.0.1.3680043.8.498.24177836893274703006130063048628967649","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272","series":"1.2.826.0.1.3680043.8.498.14940123451397990101460119404617793770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952","series":"1.2.826.0.1.3680043.8.498.80640222593377212440703793080712213187","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309","series":"1.2.826.0.1.3680043.8.498.41233498997007610529965212975325922377","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80876235930771882271600826613593202364","series":"1.2.826.0.1.3680043.8.498.36963924098164124291962746354396780169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834","series":"1.2.826.0.1.3680043.8.498.12777310391604420793409362296300277900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057","series":"1.2.826.0.1.3680043.8.498.17248753995787997622627171616633692427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81096597169632636454986988944050298701","series":"1.2.826.0.1.3680043.8.498.50122514681113464939866915784811705362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673","series":"1.2.826.0.1.3680043.8.498.52702293378333826654509806872790056792","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340","series":"1.2.826.0.1.3680043.8.498.34864942614907937080611669471033274473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.58412189477289752150471708831575518867","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694","series":"1.2.826.0.1.3680043.8.498.26052370632187729240736972703454542855","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","series":"1.2.826.0.1.3680043.8.498.46468891587567666364708707642526750480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272","series":"1.2.826.0.1.3680043.8.498.11047582596951666396593373411986735805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563","series":"1.2.826.0.1.3680043.8.498.82844793591978671980982988716722535811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.95430368480070340433845939189343814849","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81651132686162360431299931833372141617","series":"1.2.826.0.1.3680043.8.498.72323942264560964036551391339741135996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.11069097667501178788576320422006950172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81740348974208459068937862799489495782","series":"1.2.826.0.1.3680043.8.498.35623999302794596896574409601033167579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81793454171031578492222004629265937656","series":"1.2.826.0.1.3680043.8.498.12361443632100370540041561454219631943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81819209302731920404237374320026807282","series":"1.2.826.0.1.3680043.8.498.11550376991779079869320540650085228835","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.40676829279101764896145178293441368165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81957447654112577010261210572986132211","series":"1.2.826.0.1.3680043.8.498.70388006590156572680481364364695835118","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277","series":"1.2.826.0.1.3680043.8.498.74579462615203298919784496351624271027","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678","series":"1.2.826.0.1.3680043.8.498.37562623600069209795106416704593385708","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476","series":"1.2.826.0.1.3680043.8.498.10507146993605706465158309675206728735","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.10124696712778542687961602116738150466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263","series":"1.2.826.0.1.3680043.8.498.19145194351809932717630407521185912972","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82355456395534835058857499033494900716","series":"1.2.826.0.1.3680043.8.498.20935980470715702605367826269901265774","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222","series":"1.2.826.0.1.3680043.8.498.31841346645254465864900276143259536766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874","series":"1.2.826.0.1.3680043.8.498.44647321354926713240928820629284272493","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82548952153746644951736669119607083230","series":"1.2.826.0.1.3680043.8.498.13205432412760628896352215608391269928","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700","series":"1.2.826.0.1.3680043.8.498.12480021517751959907459532628532701019","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714","series":"1.2.826.0.1.3680043.8.498.67599725324612311395293453090935170191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758","series":"1.2.826.0.1.3680043.8.498.11388188159839058260690075296818889863","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400","series":"1.2.826.0.1.3680043.8.498.26327921072813517273596905408835438004","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717","series":"1.2.826.0.1.3680043.8.498.11398093380944861189072894550574974268","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966","series":"1.2.826.0.1.3680043.8.498.19635739428844317290418324120194947027","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.11900821490723913589882333061912792214","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825","series":"1.2.826.0.1.3680043.8.498.28587931124195833179442797728978890548","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732","series":"1.2.826.0.1.3680043.8.498.28075865845810342330875541451813300585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.25557406479924895086794592761134442150","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638","series":"1.2.826.0.1.3680043.8.498.11186175687330955095177093642068588657","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936","series":"1.2.826.0.1.3680043.8.498.31437262134297525037228658378489443738","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462","series":"1.2.826.0.1.3680043.8.498.12666676577914024754641726981234607949","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.67029174950551394667256977933360193715","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83847285247209768917850605454140194088","series":"1.2.826.0.1.3680043.8.498.92284903959262534377860813363600468582","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532","series":"1.2.826.0.1.3680043.8.498.24402285189343822427344552619963985643","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","series":"1.2.826.0.1.3680043.8.498.11205596325080670767020317159806419381","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220","series":"1.2.826.0.1.3680043.8.498.10335368334126589533721241021848077908","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","series":"1.2.826.0.1.3680043.8.498.17417139243766922187077317601963670620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261","series":"1.2.826.0.1.3680043.8.498.28769867970355949221656866370346875103","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259","series":"1.2.826.0.1.3680043.8.498.56325040710639780797650942329798043341","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475","series":"1.2.826.0.1.3680043.8.498.10121325242022499928920409712199275450","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364","series":"1.2.826.0.1.3680043.8.498.28914888333261066413773514116632672669","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84709539128629951937294117586370340042","series":"1.2.826.0.1.3680043.8.498.74879374858550031360410965901812803360","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.18731737284379400023038810339092491053","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396","series":"1.2.826.0.1.3680043.8.498.10198678456892324234913796548982184963","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84982310626395569925821771375535259394","series":"1.2.826.0.1.3680043.8.498.13389715098236098392200944566028771000","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548","series":"1.2.826.0.1.3680043.8.498.76907418313676155021855671454670188870","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.11890482817822444234298102529104615095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622","series":"1.2.826.0.1.3680043.8.498.80303252606793377011446169877431372120","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.90443248348101182939123313144949576195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85341019006931436802294963713199308935","series":"1.2.826.0.1.3680043.8.498.74893540225092931412754222777819722942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345","series":"1.2.826.0.1.3680043.8.498.11691918524010640030201239805226601908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","series":"1.2.826.0.1.3680043.8.498.13714246045363329399525301093790927511","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327","series":"1.2.826.0.1.3680043.8.498.26324802569335008623303574195574167369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85545102350084131396266710667808696561","series":"1.2.826.0.1.3680043.8.498.28621920035825851275057241702229771102","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85602291032341007550376696695283794665","series":"1.2.826.0.1.3680043.8.498.18527120983837724164991370978745713021","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85649172620970194090967653766415467836","series":"1.2.826.0.1.3680043.8.498.56516393548738429666589817259506915163","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.68371541544754281087869738280527054693","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.65674958163967297710698966278194805045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674","series":"1.2.826.0.1.3680043.8.498.72010378616602860772861371661259511361","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","series":"1.2.826.0.1.3680043.8.498.41198028731243247341332859373802016711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863","series":"1.2.826.0.1.3680043.8.498.11231585310159684323578982353455234985","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.31284104894238188776923947684630239260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996","series":"1.2.826.0.1.3680043.8.498.72496079948989923146825270384935339872","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85992037603186775437749979929724514304","series":"1.2.826.0.1.3680043.8.498.15127037221336345050633580968945630256","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827","series":"1.2.826.0.1.3680043.8.498.11458006570021954723770456372496894378","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308","series":"1.2.826.0.1.3680043.8.498.11816049407038575014030672113651350128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915","series":"1.2.826.0.1.3680043.8.498.38817237858687749186125650732630711043","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374","series":"1.2.826.0.1.3680043.8.498.72818731892123246955697759965748490389","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057","series":"1.2.826.0.1.3680043.8.498.70154584565198483666067331382608927417","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373","series":"1.2.826.0.1.3680043.8.498.20560230959811784922435553529037592379","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601","series":"1.2.826.0.1.3680043.8.498.10754074000520390911746317600900273675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391","series":"1.2.826.0.1.3680043.8.498.39232298351977405935117628682032535357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86422343639522250286002195622435564347","series":"1.2.826.0.1.3680043.8.498.10260448143834221138804866455725415664","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543","series":"1.2.826.0.1.3680043.8.498.27191375320070289308205936475247153446","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86566584776908143877603164704838586097","series":"1.2.826.0.1.3680043.8.498.10186228716184800802631033108256645980","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86651345701812229172985086531423369421","series":"1.2.826.0.1.3680043.8.498.94594239573250181510336139495885996254","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.13382831330123363166859944796847946545","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.72671356662664670063950888722337163335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930","series":"1.2.826.0.1.3680043.8.498.26928345788234252492313865443781476475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937","series":"1.2.826.0.1.3680043.8.498.21789120799125842983040971155713494549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","series":"1.2.826.0.1.3680043.8.498.10131822381875504921387974445796750260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633","series":"1.2.826.0.1.3680043.8.498.48079466480857694304893492556319428460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87222653734797522858506220026427900177","series":"1.2.826.0.1.3680043.8.498.35311643845122663441194118923686908064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.11888460886041601126465820090674482613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87267658933515453963224923798100651549","series":"1.2.826.0.1.3680043.8.498.45336250215728878387549358309926427853","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87413924318793242918916384453755054267","series":"1.2.826.0.1.3680043.8.498.35875030538618055791310592403555615740","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155","series":"1.2.826.0.1.3680043.8.498.86274440903199102010771814560717985537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87550037046735743504446944390018580426","series":"1.2.826.0.1.3680043.8.498.66892886796844642748309264471663044393","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.39112969435218845529032392985573641806","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.72877733719075765955506448091905512693","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839","series":"1.2.826.0.1.3680043.8.498.13066843648839739600519440796135843076","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885","series":"1.2.826.0.1.3680043.8.498.92582273756917162154117416494197851018","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035","series":"1.2.826.0.1.3680043.8.498.92800150362212319400591917341664884021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743","series":"1.2.826.0.1.3680043.8.498.31969755613319396991419662713145915461","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523","series":"1.2.826.0.1.3680043.8.498.73074532803989915022495035908684507719","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905","series":"1.2.826.0.1.3680043.8.498.18472351093496339985954624820095080423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350","series":"1.2.826.0.1.3680043.8.498.11349248352396670247514150745011086712","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894","series":"1.2.826.0.1.3680043.8.498.11531290103737170784348806642174283472","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88439732365857773633852840383740478214","series":"1.2.826.0.1.3680043.8.498.90179231107161225401647833494235970465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88480827968231439729532202780922145132","series":"1.2.826.0.1.3680043.8.498.83500073757637233833378178131317511207","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080","series":"1.2.826.0.1.3680043.8.498.21220258970176325213934132186266543219","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560","series":"1.2.826.0.1.3680043.8.498.63621080065680832688612443789936383905","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88620225126852321191690360547272730103","series":"1.2.826.0.1.3680043.8.498.71596365947627162105121308046361395060","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184","series":"1.2.826.0.1.3680043.8.498.56991018812719758558376566712714876391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928","series":"1.2.826.0.1.3680043.8.498.31097559556870627364456223996564484660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763","series":"1.2.826.0.1.3680043.8.498.10526727703711025722578792374886894488","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.68693625313866118434356556638774004302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328","series":"1.2.826.0.1.3680043.8.498.10032884996285576884174467411755805461","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88874647535130028956725641820977918615","series":"1.2.826.0.1.3680043.8.498.13254261695759851118126436877001013858","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067","series":"1.2.826.0.1.3680043.8.498.11138742229861951560726191525492710819","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.65297547223639681711818565897803744951","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.18518449639016386818141771375036994885","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.32615236646542211674664053681939413388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89204062904097086723738217311755833877","series":"1.2.826.0.1.3680043.8.498.70877026776535695778977806185862550167","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186","series":"1.2.826.0.1.3680043.8.498.38238576165806717141425022462844311975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530","series":"1.2.826.0.1.3680043.8.498.23168681508684889671901126985402673008","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.93771350695620962039711731848975140246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89416235692252764852783739140428958560","series":"1.2.826.0.1.3680043.8.498.92660380589143744022767259638950679258","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903","series":"1.2.826.0.1.3680043.8.498.56275008757888635867515048921113499497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927","series":"1.2.826.0.1.3680043.8.498.13351211023082680684452529380740566399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","series":"1.2.826.0.1.3680043.8.498.43792086070961494398103561576077977142","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.11028439450767729022506106473499940119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89842390998700046088751402253903762827","series":"1.2.826.0.1.3680043.8.498.19710845407907921356552099633890644356","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722","series":"1.2.826.0.1.3680043.8.498.59153231910775150158900784108086334622","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011","series":"1.2.826.0.1.3680043.8.498.10416011950032441927909450812059974883","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280","series":"1.2.826.0.1.3680043.8.498.14959447845105368561302643209824643285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90103210546658057701715546323552886913","series":"1.2.826.0.1.3680043.8.498.73421617837244897866669505855156954170","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90170179734833731758923964833173536172","series":"1.2.826.0.1.3680043.8.498.70311608318332669864100680569233079597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90206716031776606956011363950625953310","series":"1.2.826.0.1.3680043.8.498.41124652549831788543407991222988913015","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.13247872358623783695590224310691643825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876","series":"1.2.826.0.1.3680043.8.498.74548603086688078220647715590270403349","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767","series":"1.2.826.0.1.3680043.8.498.40976673889922888495356759133819064735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","series":"1.2.826.0.1.3680043.8.498.14825369532658473697632889010354474927","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90342120331229511020327052213821820086","series":"1.2.826.0.1.3680043.8.498.51779113489491021539670283757171814246","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90446842983860370183814559926724769347","series":"1.2.826.0.1.3680043.8.498.30265542868485822496910093647979074773","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410","series":"1.2.826.0.1.3680043.8.498.12164124413530218041950279641527749466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.45124647116330655844115942870681640148","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.93134319033742687879363128441345815551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240","series":"1.2.826.0.1.3680043.8.498.79060427398123161522643379541137483525","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839","series":"1.2.826.0.1.3680043.8.498.10986250178618429155806867590614167669","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90771070773751767226565553195436971014","series":"1.2.826.0.1.3680043.8.498.12862469747177209185590022989175937982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296","series":"1.2.826.0.1.3680043.8.498.47813175062837617065980399507987319239","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573","series":"1.2.826.0.1.3680043.8.498.98547477536696280812107166261089851289","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.41832713334397391238535772998980062045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","series":"1.2.826.0.1.3680043.8.498.47247725856425423337894431299692969524","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528","series":"1.2.826.0.1.3680043.8.498.10205047095187433283502066935718454128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811","series":"1.2.826.0.1.3680043.8.498.51898352210738833278421662278178219804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765","series":"1.2.826.0.1.3680043.8.498.41604581243869963814779875511939289555","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.11581357038201549313736784732149663273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042","series":"1.2.826.0.1.3680043.8.498.10430834601457865128914957239772621676","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785","series":"1.2.826.0.1.3680043.8.498.11348837319065726293868690642830360104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579","series":"1.2.826.0.1.3680043.8.498.13360624768480457133224674087460640395","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91606395836371290680039618164703651751","series":"1.2.826.0.1.3680043.8.498.60145168201406346160967855134996787994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91686669332744972709173289374472094988","series":"1.2.826.0.1.3680043.8.498.10108369068950896749966333655444975836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91744384920314309659802891351055961910","series":"1.2.826.0.1.3680043.8.498.11401132954191968210370808666230655777","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593","series":"1.2.826.0.1.3680043.8.498.93774557287685087912999084216491076266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91910585599088493175245713843727208369","series":"1.2.826.0.1.3680043.8.498.31593624292450691160487348731377302911","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395","series":"1.2.826.0.1.3680043.8.498.69684383699796665852377059310338342970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069","series":"1.2.826.0.1.3680043.8.498.23518730942437849507634068172026961805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494","series":"1.2.826.0.1.3680043.8.498.79020048274269852811414874400901717711","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675","series":"1.2.826.0.1.3680043.8.498.10344580960135488484991066625376842810","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671","series":"1.2.826.0.1.3680043.8.498.11560961549203372192371354832108194781","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728","series":"1.2.826.0.1.3680043.8.498.13108125610102826319317251512708374037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.11336613741310546076404814266244472577","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92455257597708674644536154732642099964","series":"1.2.826.0.1.3680043.8.498.11964731747808628940769183093670689769","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92597065276819676534178545455668171085","series":"1.2.826.0.1.3680043.8.498.11809871210135789320915306474615393979","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","series":"1.2.826.0.1.3680043.8.498.11306295453589938021990765462368989820","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.99198191456957743037623873366015118798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92722393862174380736027353110424417588","series":"1.2.826.0.1.3680043.8.498.19884494359878362323046636846685674556","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401","series":"1.2.826.0.1.3680043.8.498.24418143209545565410439845299592443114","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.11945432383342102965348605833257124787","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","series":"1.2.826.0.1.3680043.8.498.11779884783711324637739887142009291946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956","series":"1.2.826.0.1.3680043.8.498.97955600121512273212402524790375127343","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","series":"1.2.826.0.1.3680043.8.498.63434427953965233013527685036491853744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963","series":"1.2.826.0.1.3680043.8.498.12545939227716719278882762791281976958","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032","series":"1.2.826.0.1.3680043.8.498.10981920078136634510359344600556169532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.12397239377453170491328723459532084884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679","series":"1.2.826.0.1.3680043.8.498.10657284404100076066130152737129537521","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271","series":"1.2.826.0.1.3680043.8.498.11433906251449407241842407953085675364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068","series":"1.2.826.0.1.3680043.8.498.10980920712746219305359095743550414185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93728117237268000947508816107096388086","series":"1.2.826.0.1.3680043.8.498.11160584508249655973007852775249068784","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","series":"1.2.826.0.1.3680043.8.498.17755933510106554626638699448574974065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064","series":"1.2.826.0.1.3680043.8.498.35848921178565317155494721569992420225","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401","series":"1.2.826.0.1.3680043.8.498.78519134260947533574775363889451484594","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94030816415316308210133175314873817171","series":"1.2.826.0.1.3680043.8.498.27613905917808587834942798487157462597","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475","series":"1.2.826.0.1.3680043.8.498.54432398805474937978935731693377674356","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94255124739558547668304022649099251744","series":"1.2.826.0.1.3680043.8.498.25802652950504398801646132082913234631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94328216642444410827846344286227627425","series":"1.2.826.0.1.3680043.8.498.89394238258705182502147352592751216038","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94381282809153270156987123429739805206","series":"1.2.826.0.1.3680043.8.498.17413421904431497952087005536054741358","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413","series":"1.2.826.0.1.3680043.8.498.11906610033468120117486149388205741382","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.10172215480709509782568654919519753034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304","series":"1.2.826.0.1.3680043.8.498.24925908088493183988657039031208336533","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740","series":"1.2.826.0.1.3680043.8.498.11729263015298923297339264779900740608","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618","series":"1.2.826.0.1.3680043.8.498.35266027561022999626509132375714014276","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94747403767627336827372096100476526821","series":"1.2.826.0.1.3680043.8.498.60164901574588007435955838467600044567","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511","series":"1.2.826.0.1.3680043.8.498.39620999095852288371302531027338894640","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764","series":"1.2.826.0.1.3680043.8.498.12016592084133249981349775278268625508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.11402094225973310242779488202591581754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224","series":"1.2.826.0.1.3680043.8.498.11067813189509868854827558880905089851","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95030266788619257057351879256576338162","series":"1.2.826.0.1.3680043.8.498.31370233247028878230228751834693544438","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308","series":"1.2.826.0.1.3680043.8.498.29140728774502280602335429582215835064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318","series":"1.2.826.0.1.3680043.8.498.63270633372361767201067851098078651295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.12475471328262859038820261930081455843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139","series":"1.2.826.0.1.3680043.8.498.40816176723814572770899198433517509852","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297","series":"1.2.826.0.1.3680043.8.498.91907687764799591889428845744135442364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","series":"1.2.826.0.1.3680043.8.498.91506092194277450482480258478861352661","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815","series":"1.2.826.0.1.3680043.8.498.12198710533514872571366919780751400133","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","series":"1.2.826.0.1.3680043.8.498.84649131267758914731312157740785557003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","series":"1.2.826.0.1.3680043.8.498.81686033957801490188294303825951394421","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863","series":"1.2.826.0.1.3680043.8.498.97652346448936203414037892981930118276","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95621693348567815920675804584086926261","series":"1.2.826.0.1.3680043.8.498.11771293638880155004283215395248649524","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180","series":"1.2.826.0.1.3680043.8.498.20609760135342871782131179982182567631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247","series":"1.2.826.0.1.3680043.8.498.79797119996879893771626825369470988124","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","series":"1.2.826.0.1.3680043.8.498.11179604509598574563663201862422647122","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078","series":"1.2.826.0.1.3680043.8.498.15883051124256861363159912164108640600","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95912798959818676102590029534562878721","series":"1.2.826.0.1.3680043.8.498.23283515884269096856764925911010176972","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466","series":"1.2.826.0.1.3680043.8.498.11594307268832409644859362083140791178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871","series":"1.2.826.0.1.3680043.8.498.63585078602995115921993816958110800506","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.11723643942399433530065452503853679042","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.66075606903930517333169099258422286863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96316229168827165128604998083731481211","series":"1.2.826.0.1.3680043.8.498.11250407329977938614020112935106189711","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420","series":"1.2.826.0.1.3680043.8.498.22392208934347993433497022038927266747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96510621665552630060821422211995212854","series":"1.2.826.0.1.3680043.8.498.20146609910587671256656776328801427826","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.11405738615098393538228748005854902835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083","series":"1.2.826.0.1.3680043.8.498.15280867700975953278833266586239718947","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96621557839848412555023112236627663339","series":"1.2.826.0.1.3680043.8.498.12237747273148367452322201436765693941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","series":"1.2.826.0.1.3680043.8.498.10428446043070032107590589193895865706","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96943946564115188893954628924857155670","series":"1.2.826.0.1.3680043.8.498.75036634422845267406535562407402968806","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883","series":"1.2.826.0.1.3680043.8.498.19073144068136778690067287317155653068","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623","series":"1.2.826.0.1.3680043.8.498.70090773720439964590420889071497856339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556","series":"1.2.826.0.1.3680043.8.498.27475632477218177901184338471312925795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97107472223824971578324157203273471706","series":"1.2.826.0.1.3680043.8.498.62576763035455152242094652088356594706","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97156518925820521351080132062400795929","series":"1.2.826.0.1.3680043.8.498.51041715661091745986913288320057074294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.58877976317578758398442088136405526052","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97259977321685658337087695233264517545","series":"1.2.826.0.1.3680043.8.498.11717834861022626656324325174688228388","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97274847570547462056218828853253311223","series":"1.2.826.0.1.3680043.8.498.93148878122977282525884038282615509621","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684","series":"1.2.826.0.1.3680043.8.498.13230031280046555570733783115418187459","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504","series":"1.2.826.0.1.3680043.8.498.11579815396611340616177892495865883162","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","series":"1.2.826.0.1.3680043.8.498.26828897492236540057121610853830513339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.42296261557446748621275141358886404309","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527","series":"1.2.826.0.1.3680043.8.498.87171684004985397055387071703600433743","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028","series":"1.2.826.0.1.3680043.8.498.10363616349203392356541213139184247318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97694479664965569386338738579573728752","series":"1.2.826.0.1.3680043.8.498.78287146275449387674832399693485284134","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614","series":"1.2.826.0.1.3680043.8.498.44817545497803689736438377387228762222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668","series":"1.2.826.0.1.3680043.8.498.11118173267239993171700766318845288335","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933","series":"1.2.826.0.1.3680043.8.498.64905692565828457961323398047920752165","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086","series":"1.2.826.0.1.3680043.8.498.11024154326156570705450480333789735027","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187","series":"1.2.826.0.1.3680043.8.498.15493835729943930078196661947854679712","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381","series":"1.2.826.0.1.3680043.8.498.13157344127524195577913855068918142676","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974","series":"1.2.826.0.1.3680043.8.498.16254664491144244871000664116832631289","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98112373403331748422888362310572528002","series":"1.2.826.0.1.3680043.8.498.10869852919842692576102074148319508715","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98151095279541278201351784048265686573","series":"1.2.826.0.1.3680043.8.498.71572247295773000804881931492365621792","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","series":"1.2.826.0.1.3680043.8.498.18872830193444225070287045339676795124","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.54769010889792163797485985524342014971","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287","series":"1.2.826.0.1.3680043.8.498.12518984357265256809227906755565766807","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.46948743524877746104571154541975538055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070","series":"1.2.826.0.1.3680043.8.498.54770350819116488838733827086706752495","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625","series":"1.2.826.0.1.3680043.8.498.19897622745201535671510733894367827839","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","series":"1.2.826.0.1.3680043.8.498.10782856832358981527959285470269436699","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","series":"1.2.826.0.1.3680043.8.498.75784386872752929896632877182029486814","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366","series":"1.2.826.0.1.3680043.8.498.55344693187628528237068788895266607011","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874","series":"1.2.826.0.1.3680043.8.498.67162477212545716527885916902640772850","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99088247184917460283293617180083591028","series":"1.2.826.0.1.3680043.8.498.30391597261088276593154511798252105084","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201","series":"1.2.826.0.1.3680043.8.498.14228217647786167786579054765076886602","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.24297466937188030770025456540752395309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99309981642816644502709148844743995902","series":"1.2.826.0.1.3680043.8.498.11939568846336015508658969139211680875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.10189938997770718878462551187492288229","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280","series":"1.2.826.0.1.3680043.8.498.26283827235516621111588239380119001761","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928","series":"1.2.826.0.1.3680043.8.498.11772041002043141791675801772871443841","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961","series":"1.2.826.0.1.3680043.8.498.34646978966919327007874770608641230938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095","series":"1.2.826.0.1.3680043.8.498.87290613282201687267019846861744769872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036","series":"1.2.826.0.1.3680043.8.498.17911902705430423696018460677575809770","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99726731706411379395165830480890405115","series":"1.2.826.0.1.3680043.8.498.12474101104915585977900417425292325269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934","series":"1.2.826.0.1.3680043.8.498.12411678072635866134288058085427599742","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99864345386635409953374818045984558970","series":"1.2.826.0.1.3680043.8.498.32924482106953812582918145667303011423","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290","series":"1.2.826.0.1.3680043.8.498.28749327090880354585823465051063842999","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973","series":"1.2.826.0.1.3680043.8.498.54980506759743668422859724813549869128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
